# Gen_AI — VAE vs DDPM from Scratch (Kaggle entry point)

**Task code:** GenCV003 · **Position:** CV Engineer

Materializes the full `Gen_AI` professional repo structure via `%%writefile`,
then runs `train.py`, `evaluate.py`, `sample.py`. This notebook and the
GitHub repo are byte-for-byte the same code.

**Setup before running:**
1. Settings -> Accelerator -> GPU.
2. "Add Input" -> search "CelebFaces Attributes (CelebA) Dataset" -> Add.
3. Run all cells top to bottom.

In [ ]:
!pip install -q torchmetrics torch-fidelity scikit-learn imageio

In [ ]:
import os
os.makedirs("/kaggle/working/Gen_AI", exist_ok=True)
os.chdir("/kaggle/working/Gen_AI")
for d in ["configs", "src/datasets", "src/models", "src/losses", "src/training",
          "src/sampling", "src/evaluation", "src/visualization", "src/utils",
          "src/engine", "tests", "outputs"]:
    os.makedirs(d, exist_ok=True)
print(os.getcwd())

## 1. Write the project files (identical to the GitHub repo)

In [ ]:
%%writefile CHANGELOG.md
# Changelog

All notable changes to this project are documented here. Format loosely
follows [Keep a Changelog](https://keepachangelog.com/).

## v1.2

**Added**
- Cosine beta schedule (Nichol & Dhariwal, "Improved DDPM", 2021) as the new
  default, alongside the original linear schedule (`configs/diffusion.py`:
  `beta_schedule`). Verified the cosine schedule degrades signal more
  gradually than linear (85% signal retained at t=250 vs. 52% for linear).
- Self-attention at an additional U-Net resolution (16x16), not just the 8x8
  bottleneck (`configs/model.py`: `unet_attn_resolutions`). Verified
  attention modules land at exactly the intended resolution level, both
  down and up paths, via a direct architecture inspection test.
- Gradient clipping for DDPM training (`configs/training.py`:
  `ddpm_grad_clip_norm`, default max norm 1.0). Correctly unscales AMP
  gradients before clipping.
- Cosine learning-rate decay over DDPM training
  (`configs/training.py`: `ddpm_use_lr_scheduler`).

**Fixed**
- **LR scheduler + resume interaction.** Resuming training with a different
  total epoch count than the original run (a normal part of this project's
  workflow) corrupted the cosine LR curve, because `CosineAnnealingLR`
  computes each step recursively from the *current* optimizer LR rather than
  a fixed base value — and the resumed optimizer's LR was already decayed
  near zero from the previous run, so fast-forwarding multiplied zero by
  zero forever. Fixed by resetting to the true base LR before fast-forwarding
  the scheduler to the correct point on the new curve. Verified the resulting
  LR sequence matches the exact theoretical cosine formula.
- **U-Net checkpoint loading crash.** `evaluate.py`, `sample.py`, and
  `src/evaluation/evaluator.py` manually reconstructed the U-Net for
  inference without passing `img_size`/`attn_resolutions`, so the rebuilt
  model didn't match the trained checkpoint's architecture the moment
  attention-resolution support was added (`RuntimeError: Unexpected key(s)
  in state_dict`). Fixed all three call sites to use `UNet.from_config(cfg)`
  consistently instead of repeating constructor arguments by hand.

**Results**

| Metric | v1.1 | v1.2 | Change |
|---|---|---|---|
| VAE FID | 97.94 | 97.06 | -0.9% (flat, expected — no VAE changes this version) |
| VAE Inception Score | 2.00 +/- 0.08 | 2.00 +/- 0.06 | flat |
| VAE reconstruction PSNR | 22.48 dB | 22.47 dB | flat |
| DDPM FID | 28.37 | 26.42 | -6.9% |
| DDPM Inception Score | 2.59 +/- 0.13 | 2.61 +/- 0.17 | ~flat |

Smaller gain than v1.0->v1.1, as expected: v1.1's improvement included fixing
an outright sampling bug (large, one-time win), while v1.2 is architectural
refinement on an already-working baseline (smaller, incremental win).

**Artifacts:** `outputs/v1.2/results.json`,
`outputs/comparison/version_comparison.png` (cross-version FID/IS chart,
regenerate with `python scripts/generate_comparison_chart.py`)

---

## v1.1

**Fixed**
- **DDPM ancestral sampler numerical drift.** `src/sampling/ddpm_sampler.py`
  never clipped the intermediate predicted-x0 estimate back to the valid
  `[-1, 1]` pixel range at each reverse step. Over 1000 sequential steps,
  small per-step numerical error compounded — verified directly by comparing
  sampler output with clipping disabled: pixel values reached **-6.06 to
  6.72** (should be within [-1, 1]). This produced solid black/white panels
  in place of faces in some v1.0 samples.
  - Fix: predict x0 from the model's noise estimate at each step, clip it to
    `[-1, 1]`, then compute the posterior mean from the clipped x0 (standard
    approach from the official DDPM / Improved-DDPM implementations).
  - Added `posterior_mean_coef1` / `posterior_mean_coef2` to
    `src/models/diffusion.py`'s `GaussianDiffusion` to support this.
  - This fix applies to sampling only — it required no retraining, only
    re-running generation with the corrected sampler against existing
    checkpoints.
- **matplotlib backend leakage in interactive notebooks.** Several
  visualization modules (`sampling_grid.py`, `training_curves.py`,
  `reconstruction.py`, `interpolation.py`, `latent_space.py`,
  `denoising.py`) called `matplotlib.use("Agg")` at import time. In a Kaggle
  notebook, importing any of these silently switched the *entire session's*
  matplotlib backend to Agg, which made all subsequent `plt.show()` calls
  (including the user's own plotting cells) render nothing with no error.
  Fix: removed the forced backend — `plt.savefig()` works correctly on
  whichever backend is already active, so forcing one wasn't necessary.
- Checkpoint resume was missing `history` in the saved state dict, which
  would have raised `KeyError: 'history'` on resume. Caught by an explicit
  interrupt-and-resume test, not just an import/syntax check. Fixed in
  `src/training/train_vae.py` and `src/training/train_ddpm.py`.

**Changed**
- Dataset subset size: 30,000 -> 40,000 images
- VAE training: 30 -> 40 epochs
- DDPM training: 40 -> 50 epochs

**Results**

| Metric | v1.0 | v1.1 | Change |
|---|---|---|---|
| VAE FID | 120.69 | 97.94 | -18.9% |
| VAE Inception Score | 2.04 +/- 0.06 | 2.00 +/- 0.08 | ~flat |
| VAE reconstruction PSNR | 22.27 dB | 22.48 dB | ~flat |
| DDPM FID | 43.77 | 28.37 | -35.2% |
| DDPM Inception Score | 2.79 +/- 0.12 | 2.59 +/- 0.13 | slightly lower |

Note: this release bundles the sampler fix with a larger training budget in
a single run — the FID improvement reflects both, not the fix in isolation.

**Artifacts:** `outputs/v1.1/` (results.json, sample grids, comparison grid,
reconstruction grid, latent-space PCA, interpolation grid, denoising GIF)

---

## v1.0

Initial release: from-scratch VAE and DDPM implementation, professional
modular repo structure (`configs/`, `src/{datasets,models,losses,training,
sampling,evaluation,visualization,utils,engine}`, `tests/`).

- Dataset: CelebA, 30,000-image subset
- VAE: 30 epochs, latent_dim=128, conv encoder/decoder
- DDPM: 40 epochs, U-Net (base_ch=64, attention at 8x8 bottleneck only),
  linear beta schedule (1e-4 -> 0.02), 1000 timesteps, EMA (decay=0.999)
- Ancestral DDPM sampling only (no DDIM yet at this point in development)
- Resumable checkpointing (model, optimizer, AMP scaler, EMA state) every 2
  epochs

**Results**

| Metric | Value |
|---|---|
| VAE FID | 120.69 |
| VAE Inception Score | 2.04 +/- 0.06 |
| VAE reconstruction PSNR | 22.27 dB |
| DDPM FID | 43.77 |
| DDPM Inception Score | 2.79 +/- 0.12 |

**Known issue at this version:** some DDPM samples rendered as solid
black/white panels due to the sampler numerical drift described above (fixed
in v1.1).

**Artifacts:** `outputs/v1.0/` (results.json, sample grids, comparison grid)

---

## Planned (v1.3, not yet implemented)

Organized by which part of the codebase each change touches.

**VAE (`src/models/vae.py`, `src/losses/vae_loss.py`) — currently unchanged
since v1.0, all v1.1/v1.2 work targeted DDPM:**
- Enable the already-implemented perceptual loss
  (`configs/training.py`: `use_perceptual_loss`, currently off by default) —
  directly targets the VAE's main weakness (blur), since it compares
  reconstructions in a pretrained feature space instead of raw pixels.
- Consider a learned (rather than fixed) decoder variance, or a larger
  latent dimension — both documented ways to reduce reconstruction blur
  further.

**DDPM sampling (`src/sampling/`):**
- Benchmark the DDIM sampler (already implemented,
  `cfg.diffusion.sampler = "ddim"`) — no `results.json` has been generated
  with it yet. Worth reporting as a speed/quality trade-off data point
  (~50 steps vs. 1000) alongside the full-DDPM numbers.

**DDPM architecture (`src/models/unet.py`):**
- Try attention at 32×32 as well, or a wider U-Net (`unet_base_ch=128`) —
  the v1.1→v1.2 attention-resolution change gave a real but modest gain;
  worth checking whether more capacity continues that trend or plateaus.
- v-prediction parameterization (predicting a velocity term instead of raw
  noise) — used in several post-DDPM papers, reported to improve stability
  at high timesteps.

**Data / training budget (`configs/dataset.py`, `configs/training.py`):**
- Full ~200k CelebA images (currently using a 40k subset) and more epochs,
  now that the architecture and training loop are stable and bug-free.
- Larger image resolution (e.g. 128×128) — would require re-tuning the
  U-Net's channel multipliers and attention resolutions.

**Evaluation (`src/evaluation/`):**
- Add LPIPS or SSIM alongside FID/IS/PSNR for a more complete reconstruction-
  quality picture.
- Compute metrics separately per CelebA attribute (e.g. by gender, glasses,
  smiling) to check whether either model has attribute-specific weaknesses
  the aggregate FID hides.

**Engineering (repo-wide):**
- GitHub Actions workflow to run `pytest` automatically on every push —
  currently the test suite is only run manually.
- Replace print-based logging with TensorBoard or Weights & Biases for
  richer training-curve tracking across versions.

In [ ]:
%%writefile README.md
# Gen_AI — VAE vs DDPM from Scratch

**Task code:** GenCV003 · **Position:** CV Engineer

A from-scratch PyTorch implementation of a **Variational Autoencoder (VAE)**
and a **Denoising Diffusion Probabilistic Model (DDPM)**, trained on CelebA
and benchmarked quantitatively (FID, Inception Score, reconstruction PSNR)
and qualitatively (sample grids, latent-space visualization, interpolation,
denoising-process animation). Both models are built from base `torch.nn`
primitives only — no pretrained VAE, no `diffusers` library.

---

## Table of contents

- [Dataset](#dataset)
- [Architecture](#architecture)
- [Project structure](#project-structure)
- [Setup](#setup)
- [How to run](#how-to-run)
- [Version comparison: v1.0 -> v1.1 -> v1.2](#version-comparison-v10---v11---v12)
- [Results](#results)
- [Report notes](#report-notes)
- [Known limitations / next steps](#known-limitations--next-steps)

---

## Dataset

[CelebFaces Attributes (CelebA) Dataset](https://www.kaggle.com/datasets/jessicali9530/celeba-dataset)
— 64×64 aligned/cropped face crops, loaded via Kaggle's dataset picker.

CelebA is a good fit for this specific comparison: large enough to be a
meaningful benchmark, simple enough (aligned, centered faces) for both models
to converge within a single Kaggle GPU session, and the VAE-vs-DDPM
blur/sharpness trade-off is very visible on faces — making the qualitative
comparison easy to see, not just a number in a table.

## Architecture

| | VAE | DDPM |
|---|---|---|
| **Core idea** | Encode to a low-dimensional Gaussian latent, decode back | Learn to reverse a fixed Markov noising process |
| **Objective** | Reconstruction (MSE) + KL divergence to N(0,I) | Noise-prediction MSE (`L_simple`, Ho et al. 2020) |
| **Generation** | 1 decoder forward pass | `T` (1000) sequential U-Net forward passes |
| **Building blocks** | Conv encoder/decoder (`src/models/encoder.py`, `decoder.py`) | U-Net with residual blocks + self-attention + sinusoidal time embedding (`src/models/unet.py`, `attention.py`) |
| **Sampling options** | Latent prior sampling (`src/sampling/latent_sampler.py`) | Full DDPM (`ddpm_sampler.py`) or fast DDIM, ~10-20x fewer steps (`ddim_sampler.py`) |

## Project structure

```
Gen_AI/
├── README.md
├── CHANGELOG.md              # full version history, see below
├── requirements.txt
├── pytest.ini
├── train.py                  # Train models from the command line
├── sample.py                 # Generate new samples
├── evaluate.py                # Evaluate trained models
├── inference.py                # Run inference on custom images
│
├── configs/
│   ├── dataset.py             # Dataset configuration
│   ├── model.py                # Model hyperparameters
│   ├── diffusion.py             # Diffusion settings
│   ├── training.py               # Training configuration
│   └── config.py                  # Combines all configurations
│
├── src/
│   ├── datasets/
│   │   ├── preprocessing.py    # Prepare dataset paths and metadata
│   │   ├── transforms.py        # Image preprocessing
│   │   ├── augmentations.py      # Data augmentation
│   │   ├── dataset.py             # PyTorch dataset
│   │   └── dataloader.py           # DataLoader creation
│   │
│   ├── models/
│   │   ├── encoder.py           # VAE encoder
│   │   ├── decoder.py            # VAE decoder
│   │   ├── vae.py                 # Variational Autoencoder
│   │   ├── attention.py            # Self-attention block
│   │   ├── unet.py                  # U-Net backbone
│   │   └── diffusion.py              # Diffusion process (noise schedule)
│   │
│   ├── losses/
│   │   ├── vae_loss.py           # VAE loss
│   │   ├── diffusion_loss.py      # Diffusion loss
│   │   └── perceptual_loss.py      # Optional perceptual loss (VGG-based)
│   │
│   ├── training/
│   │   ├── ema.py                # EMA updates
│   │   ├── optimizer.py           # Optimizer setup
│   │   ├── scheduler.py            # Learning-rate scheduler
│   │   ├── callbacks.py             # Training callbacks (checkpoint, sample preview)
│   │   ├── trainer.py                # Base trainer (resume/checkpoint logic)
│   │   ├── train_vae.py               # VAE training
│   │   └── train_ddpm.py               # DDPM training
│   │
│   ├── sampling/
│   │   ├── latent_sampler.py      # Sample latent vectors (VAE)
│   │   ├── ddpm_sampler.py         # Full ancestral DDPM sampling
│   │   ├── ddim_sampler.py          # Fast DDIM sampling
│   │   └── image_sampler.py          # Unified sampling interface
│   │
│   ├── evaluation/
│   │   ├── fid.py                 # FID metric
│   │   ├── quality.py              # Inception Score
│   │   ├── reconstruction.py        # Reconstruction evaluation (VAE only)
│   │   └── evaluator.py              # Evaluation pipeline orchestrator
│   │
│   ├── visualization/
│   │   ├── sampling_grid.py       # Sample grids
│   │   ├── training_curves.py      # Training plots
│   │   ├── reconstruction.py        # Reconstruction results
│   │   ├── interpolation.py          # Latent interpolation
│   │   ├── latent_space.py            # Latent-space PCA visualization
│   │   ├── denoising.py                # Denoising process strip
│   │   └── gif.py                       # Animation generation
│   │
│   ├── utils/
│   │   ├── seed.py                # Random seed utilities
│   │   ├── device.py               # Device selection
│   │   ├── helpers.py               # Common helpers
│   │   ├── image_utils.py            # Image tensor conversion utilities
│   │   ├── checkpoint.py              # Checkpoint save/load
│   │   ├── paths.py                    # Project paths
│   │   └── logger.py                    # Logging utilities
│   │
│   └── engine/
│       ├── trainer.py             # Training orchestrator (thin — no duplicate logic)
│       ├── evaluator.py            # Evaluation orchestrator
│       └── inferencer.py            # Inference orchestrator
│
├── scripts/
│   └── generate_comparison_chart.py  # Regenerates the cross-version FID/IS chart
│
├── notebooks/
│   └── kaggle_run.ipynb          # Kaggle entry point
│
├── tests/                         # Unit tests (pytest)
│
└── outputs/                       # Generated outputs (mostly gitignored —
                                       see below for the exception)
```

**Design note on `engine/`:** it does not duplicate `training/` or
`evaluation/` — it only orchestrates them (builds the dataloader once, calls
`train_vae` then `train_ddpm`, loads checkpoints for evaluation/inference).
All actual logic lives in exactly one place.

**Note on `outputs/`:** checkpoints, logs, and samples are gitignored (too
large for git), but `outputs/v1.0/`, `v1.1/`, `v1.2/`, and
`outputs/comparison/` are explicitly kept — they hold each version's
`results.json` and comparison charts referenced throughout this README.

## Setup

```bash
pip install -r requirements.txt
```

Requires Python 3.10+, PyTorch 2.x, `torchmetrics` + `torch-fidelity` (FID/IS),
`scikit-learn` (latent-space PCA), `imageio` (denoising GIF).

## How to run

### On Kaggle

1. New Notebook → Settings → Accelerator → GPU (T4 x2 or P100).
2. "Add Input" → search **CelebFaces Attributes (CelebA) Dataset** → Add.
3. Open `notebooks/kaggle_run.ipynb`, paste its cells in order. They write
   this exact file tree via `%%writefile`, then run:

   ```bash
   python train.py
   python evaluate.py
   ```

4. If your dataset mounts at a non-standard path, check
   `src/datasets/preprocessing.py`'s `CELEBA_DIR_CANDIDATES` — add your exact
   path if `train.py` raises `FileNotFoundError`.

### Installation

```bash
pip install -r requirements.txt
```

### Training

```bash
python train.py
```

Trains the VAE first, then the DDPM. Training automatically **resumes** from
the latest checkpoint if one exists (model, optimizer, AMP scaler, and EMA
state are all restored) — safe to re-run after an interrupted session.

### Evaluation

```bash
python evaluate.py
```

Computes FID, Inception Score, and reconstruction metrics, and generates all
visualizations (sample grids, latent-space PCA, interpolation, denoising GIF).

### Sampling

```bash
python sample.py --model ddpm --n 16
```

Generates 16 new images using the selected sampler
(`--model vae`, `--model ddpm`, or `--model both`).

### Inference

Reconstruct an input image:

```bash
python inference.py --reconstruct path/to/image.jpg
```

Generate a new image:

```bash
python inference.py --generate ddpm
```

### Tests

```bash
pytest
```

Runs the project's test suite (model shapes, loss values, sampler output,
dataset pipeline).

---

## Version comparison: v1.0 -> v1.1 -> v1.2

Three full training runs were completed. Each version bundles more than one
change, and the tables below are transparent about that rather than
attributing improvements to a single factor in isolation.

![FID and Inception Score across versions](outputs/comparison/version_comparison.png)

| | **v1.0** | **v1.1** | **v1.2** |
|---|---|---|---|
| Dataset subset | 30,000 images | 40,000 images | 40,000 images |
| VAE epochs | 30 | 40 | 40 |
| DDPM epochs | 40 | 50 | 50 |
| Beta schedule | linear | linear | **cosine** |
| DDPM sampler | ancestral, **no x0 clipping** | ancestral, **x0 clipped to [-1,1] each step** | ancestral, x0 clipped |
| U-Net attention | bottleneck (8x8) only | bottleneck (8x8) only | bottleneck (8x8) **+ 16x16** |
| Gradient clipping | none | none | **max norm 1.0** |
| LR schedule | constant | constant | **cosine decay** |

### What changed at each step

**v1.0 -> v1.1 — bug fix + more training.** The v1.0 ancestral DDPM sampler
never clipped its intermediate predicted-x0 estimate back to the valid pixel
range. Over 1000 sequential steps, small per-step numerical error
compounded — verified directly: an unclipped run produced pixel values
ranging from **-6.06 to 6.72** (valid range is [-1, 1]), which is what caused
some v1.0 DDPM samples to render as solid black/white panels instead of
faces. v1.1 fixes this (`src/sampling/ddpm_sampler.py`) and also increases
the training budget, so the FID improvement reflects both changes combined.

**v1.1 -> v1.2 — architectural refinements, same bug-free baseline.** With
the sampling bug already fixed and the training budget unchanged, v1.2 tests
four documented DDPM quality improvements together: a cosine beta schedule
(Nichol & Dhariwal, "Improved DDPM"), self-attention at an additional
resolution (16x16, not just the 8x8 bottleneck), gradient clipping, and
cosine LR decay. Because v1.1 was already a working baseline, this step
tests refinement rather than bug-fixing — correspondingly, the gain is
smaller than v1.0->v1.1's, which is the expected pattern (large one-time
wins from fixing bugs, smaller incremental wins from architecture tuning).
None of the v1.2 changes touch the VAE, so its numbers are flat by design.

### Quantitative results

| Metric | v1.0 | v1.1 | v1.2 | v1.0->v1.2 |
|---|---|---|---|---|
| VAE FID (lower is better) | 120.69 | 97.94 | 97.06 | -19.6% |
| VAE Inception Score | 2.04 +/- 0.06 | 2.00 +/- 0.08 | 2.00 +/- 0.06 | ~flat |
| VAE reconstruction PSNR (higher is better) | 22.27 dB | 22.48 dB | 22.47 dB | ~flat |
| **DDPM FID (lower is better)** | 43.77 | 28.37 | **26.42** | **-39.6%** |
| DDPM Inception Score | 2.79 +/- 0.12 | 2.59 +/- 0.13 | 2.61 +/- 0.17 | ~flat* |

\* Inception Score is a weaker signal for a single-class domain like faces
(the underlying classifier is trained on 1000 ImageNet classes, none of
which are "face") — FID is the more meaningful metric across all three runs.

Each version's artifacts (`results.json`, sample grids where available) are
kept in `outputs/v1.0/`, `outputs/v1.1/`, `outputs/v1.2/` for direct
comparison, and `outputs/comparison/version_comparison.png` (regenerate with
`python scripts/generate_comparison_chart.py` after adding a new version) —
see [CHANGELOG.md](CHANGELOG.md) for the complete version history.

---

## Results

Final reported numbers (v1.2, current best):

| Model | FID (lower better) | Inception Score (higher better) | Reconstruction PSNR (higher better) | Sampling cost |
|-------|---------------------|-----------------------------------|----------------------------------------|----------------|
| VAE   | 97.06               | 2.00 +/- 0.06                       | 22.47 dB                                | 1 forward pass |
| DDPM  | 26.42               | 2.61 +/- 0.17                       | n/a                                      | 1000 forward passes (full DDPM) |

## Qualitative Analysis

The assignment asks for a subjective assessment covering diversity, realism,
and thematic consistency — based on directly viewing generated sample grids,
not just the FID/IS numbers above.

> **Version note:** this analysis is based on the **v1.0** sample grids
> (`outputs/v1.0/vae_grid.png`, `ddpm_grid.png`). v1.1 fixed the DDPM
> sampling bug described below, so v1.1/v1.2 samples should show
> meaningfully better thematic consistency than what's described here — this
> section should be re-examined against `outputs/v1.2/` grids once available
> and updated accordingly.

**Diversity.** The VAE's 16 samples show genuine variation in age, hair
color, and expression, with no visible mode collapse. DDPM's successful
generations are similarly varied, but 2 of 16 samples in the v1.0 grid
failed to produce a face at all (rendering as a near-blank white or near-solid
black panel), which reduces *effective* diversity — a failed generation
contributes nothing to the usable output distribution.

**Realism.** This is where the architectural difference is most visible.
Every VAE sample shares the same soft, out-of-focus quality — hair blends
into the background, eye and mouth edges are indistinct, colors look
slightly washed out. Several DDPM samples are noticeably sharper — visible
individual hair strands, photographic-looking skin texture, defined eye
catchlights — clearly exceeding any VAE sample's fidelity. DDPM v1.0 also
has realism failures beyond blur: one sample shows an unnatural, matted
texture across the whole face, qualitatively different from a blurry-but-
coherent VAE failure mode.

**Thematic consistency** (does the model reliably stay "on-topic," i.e.
produce a face at all). VAE: 16/16 samples are recognizably human faces,
even the blurriest ones. DDPM v1.0: roughly 14/16 — the blank/corrupted
panels are genuine thematic failures. This is the direct visual signature of
the sampler numerical-drift bug described in
[CHANGELOG.md](CHANGELOG.md#v11) (verified pixel values reaching -6.06 to
6.72, well outside the valid [-1,1] range) — it is a sampling-implementation
issue fixed in v1.1, not an inherent limitation of diffusion models. Worth
stating explicitly in the report: DDPM's v1.0 thematic-consistency score
understates the architecture's real capability once sampled correctly.

**Summary:** VAE prioritizes reliability (always produces *a* face) at the
cost of sharpness. DDPM v1.0 produces higher peak realism but at the cost of
occasional total failures — a trade-off that the v1.1 sampling fix
substantially closes, which the FID numbers (43.77 -> 28.37 -> 26.42) are
consistent with even though this qualitative section hasn't yet been
re-verified against the newer sample grids.



- **Architecture difference:** the VAE learns an explicit, low-dimensional
  latent distribution via an encoder/decoder trained with a
  reconstruction + KL objective; the DDPM has no explicit compressed latent
  — it learns to reverse a fixed Markov noising process, trained with a
  simple noise-prediction MSE loss.
- **Sample quality:** DDPM FID is ~3.7x better than VAE FID, consistent
  across all three training runs — matches the theoretical expectation that
  diffusion models produce sharper, more realistic samples, since each
  denoising step only has to make a small, local correction rather than
  reconstruct the whole image in one pass.
- **Speed trade-off:** VAE sampling is a single forward pass; DDPM (full)
  requires 1000 sequential passes. The DDIM sampler (`src/sampling/ddim_sampler.py`)
  narrows this gap to ~50 steps at some quality cost — set
  `cfg.diffusion.sampler = "ddim"` to use it.
- **Latent space sanity check:** `outputs/v1.1/latent_space/vae_latent_pca.png`
  and `outputs/v1.1/interpolation/vae_interpolation.png` show the VAE learned
  a smooth, meaningful latent space (no posterior collapse).
- **Denoising process:** `outputs/v1.1/gifs/ddpm_denoising.gif` makes DDPM's
  iterative refinement process visually concrete.

## Known limitations / next steps

- **Training budget is still modest** relative to published DDPM results on
  CelebA (which typically use the full ~200k images and hundreds of epochs);
  FID of 26.42 is a credible result at this budget, not a ceiling.
- **DDIM sampling hasn't been benchmarked yet** — the repo supports it
  (`cfg.diffusion.sampler = "ddim"`), but no results.json has been generated
  with it. Worth adding as a speed/quality trade-off data point.
- **VAE architecture is unchanged since v1.0.** All improvements so far
  targeted DDPM; a candidate v1.3 could apply a matching set of upgrades to
  the VAE (e.g. perceptual loss, already implemented but off by default in
  `configs/training.py`).

## Deliverables mapping

- **(a) Report:** the Results table, Version comparison section, and Report
  notes above as an outline; embed the visualization PNGs/GIF from
  `outputs/v1.2/` (or `v1.1/` where v1.2-specific images weren't generated)
  for qualitative evidence.
- **(b) GitHub repo:** this repository, structured so `git clone` +
  `pip install -r requirements.txt` + `python train.py` reproduces everything.

In [ ]:
%%writefile configs/__init__.py

In [ ]:
%%writefile configs/config.py
"""
Top-level configuration composing dataset / model / diffusion / training
sub-configs. Import `Config` and override any field, e.g.:

    from configs.config import Config
    cfg = Config()
    cfg.training.vae_epochs = 50
    cfg.dataset.subset_size = 50000
"""
from dataclasses import dataclass, field

from configs.dataset import DatasetConfig
from configs.model import ModelConfig
from configs.diffusion import DiffusionConfig
from configs.training import TrainingConfig


@dataclass
class Config:
    seed: int = 42
    output_dir: str = "outputs"

    dataset: DatasetConfig = field(default_factory=DatasetConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    diffusion: DiffusionConfig = field(default_factory=DiffusionConfig)
    training: TrainingConfig = field(default_factory=TrainingConfig)

    # Evaluation
    n_eval_samples: int = 2000
    eval_batch_size: int = 64

    # --- convenience path properties (kept in sync with output_dir) ---
    @property
    def checkpoint_dir(self) -> str:
        return f"{self.output_dir}/checkpoints"

    @property
    def samples_dir(self) -> str:
        return f"{self.output_dir}/samples"

    @property
    def logs_dir(self) -> str:
        return f"{self.output_dir}/logs"

In [ ]:
%%writefile configs/dataset.py
"""Dataset-related configuration."""
from dataclasses import dataclass


@dataclass
class DatasetConfig:
    data_dir: str = ""          # leave empty to auto-detect via src/datasets/preprocessing.py
    img_size: int = 64
    subset_size: int = 30000
    batch_size: int = 128
    num_workers: int = 2
    horizontal_flip: bool = True   # light augmentation, see src/datasets/augmentations.py

In [ ]:
%%writefile configs/diffusion.py
"""Diffusion-process configuration."""
from dataclasses import dataclass


@dataclass
class DiffusionConfig:
    timesteps: int = 1000
    beta_start: float = 1e-4          # only used when beta_schedule == "linear"
    beta_end: float = 0.02            # only used when beta_schedule == "linear"
    ema_decay: float = 0.999

    # "linear" (original DDPM) or "cosine" (Nichol & Dhariwal, "Improved DDPM",
    # 2021) — cosine adds noise more gradually, avoiding the linear schedule's
    # tendency to destroy most image information well before t=T, which wastes
    # training signal on those late, already-mostly-noise steps.
    beta_schedule: str = "cosine"

    # Sampler used at inference time: "ddpm" (all T steps, slow, matches training
    # exactly) or "ddim" (fewer steps, deterministic, ~10-20x faster, small
    # quality trade-off). See src/sampling/.
    sampler: str = "ddpm"
    ddim_steps: int = 50
    ddim_eta: float = 0.0

In [ ]:
%%writefile configs/model.py
"""Model architecture configuration (VAE + U-Net)."""
from dataclasses import dataclass, field
from typing import Tuple


@dataclass
class ModelConfig:
    img_channels: int = 3

    # VAE
    latent_dim: int = 128
    vae_base_ch: int = 64

    # U-Net (noise-prediction network for DDPM)
    unet_base_ch: int = 64
    unet_ch_mults: Tuple[int, ...] = (1, 2, 4, 4)
    unet_time_dim: int = 256
    unet_attn_heads: int = 4
    # Spatial resolutions (in pixels) at which to add self-attention in the
    # U-Net, in addition to the bottleneck (which always gets attention).
    # The original DDPM paper attends at 16x16 for similarly-sized images —
    # giving the network global context (e.g. eye symmetry) earlier than the
    # 8x8 bottleneck alone allows.
    unet_attn_resolutions: Tuple[int, ...] = (16,)

In [ ]:
%%writefile configs/training.py
"""Training-loop configuration."""
from dataclasses import dataclass


@dataclass
class TrainingConfig:
    vae_epochs: int = 30
    vae_lr: float = 2e-4
    kl_warmup_epochs: int = 5      # linear KL-annealing, reduces posterior collapse

    ddpm_epochs: int = 40
    ddpm_lr: float = 2e-4
    ddpm_grad_clip_norm: float = 1.0   # max gradient norm; stabilizes longer DDPM runs
    ddpm_use_lr_scheduler: bool = True  # cosine LR decay over ddpm_epochs

    checkpoint_every: int = 2      # epochs between resumable checkpoints
    use_amp: bool = True           # mixed precision (only active on CUDA)

    use_perceptual_loss: bool = False   # adds VGG perceptual loss to the VAE objective
    perceptual_weight: float = 0.1

In [ ]:
%%writefile evaluate.py
#!/usr/bin/env python3
"""
Run the full quantitative + qualitative evaluation suite: FID, Inception
Score, VAE reconstruction metrics, comparison grids, latent-space plot,
interpolation grid, and a DDPM denoising-process strip + GIF.

    python evaluate.py
"""
import json
import os

from configs.config import Config
from src.engine.evaluator import GenAIEvaluator
from src.evaluation.evaluator import Evaluator
from src.models.unet import UNet
from src.utils.paths import ensure_output_dirs
from src.visualization.denoising import save_denoising_strip
from src.visualization.gif import save_gif
from src.visualization.interpolation import save_interpolation_grid
from src.visualization.latent_space import save_latent_space_plot
from src.visualization.reconstruction import save_reconstruction_grid


def main():
    cfg = Config()
    ensure_output_dirs(cfg)

    evaluator = GenAIEvaluator(cfg)
    results, real_imgs, vae_imgs, ddpm_imgs = evaluator.run()
    print(json.dumps(results, indent=2))

    vae = evaluator._load_vae()
    unet, diffusion, ema = evaluator._load_ddpm()
    eval_unet = UNet.from_config(cfg).to(evaluator.device)
    ema.copy_to(eval_unet)

    print("Saving reconstruction grid...")
    save_reconstruction_grid(vae, real_imgs, evaluator.device,
                              f"{cfg.output_dir}/reconstructions/vae_reconstructions.png")

    print("Saving latent-space plot...")
    save_latent_space_plot(vae, real_imgs, evaluator.device,
                            f"{cfg.output_dir}/latent_space/vae_latent_pca.png")

    print("Saving latent interpolation grid...")
    save_interpolation_grid(vae, evaluator.device,
                             f"{cfg.output_dir}/interpolation/vae_interpolation.png")

    print("Saving DDPM denoising strip + GIF...")
    frames = save_denoising_strip(eval_unet, diffusion, cfg.dataset.img_size, evaluator.device,
                                   f"{cfg.output_dir}/denoising/ddpm_denoising_strip.png")
    save_gif(frames, f"{cfg.output_dir}/gifs/ddpm_denoising.gif")

    print("All evaluation artifacts saved to", cfg.output_dir)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile inference.py
#!/usr/bin/env python3
"""
Single-image / single-sample inference.

    python inference.py --reconstruct path/to/face.jpg
    python inference.py --generate ddpm
    python inference.py --generate vae
"""
import argparse
import os

from torchvision.utils import save_image

from configs.config import Config
from src.engine.inferencer import GenAIInferencer


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--reconstruct", type=str, default=None,
                         help="Path to an image to encode+decode through the VAE")
    parser.add_argument("--generate", choices=["vae", "ddpm"], default=None,
                         help="Generate a single new image from the given model")
    parser.add_argument("--out_dir", type=str, default="outputs/inference")
    args = parser.parse_args()

    cfg = Config()
    os.makedirs(args.out_dir, exist_ok=True)
    inferencer = GenAIInferencer(cfg)

    if args.reconstruct:
        original, recon = inferencer.reconstruct_image(args.reconstruct)
        save_image(original, os.path.join(args.out_dir, "original.png"))
        save_image(recon, os.path.join(args.out_dir, "reconstructed.png"))
        print(f"Saved original + reconstruction to {args.out_dir}")

    if args.generate:
        img = inferencer.generate_one(source=args.generate)
        path = os.path.join(args.out_dir, f"generated_{args.generate}.png")
        save_image(img, path)
        print(f"Saved generated image to {path}")

    if not args.reconstruct and not args.generate:
        parser.print_help()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile pytest.ini
[pytest]
testpaths = tests
python_files = test_*.py

In [ ]:
%%writefile requirements.txt
torch>=2.0
torchvision>=0.15
torchmetrics>=1.0
torch-fidelity>=0.3
matplotlib>=3.5
numpy>=1.23
pillow>=9.0
scikit-learn>=1.2
imageio>=2.31
pytest>=7.4

In [ ]:
%%writefile sample.py
#!/usr/bin/env python3
"""
Generate new images from the trained VAE and/or DDPM.

    python sample.py --model ddpm --n 16
    python sample.py --model vae --n 16
    python sample.py --model both --n 16
"""
import argparse
import os

import torch
from torchvision.utils import save_image

from configs.config import Config
from src.engine.evaluator import GenAIEvaluator
from src.models.unet import UNet
from src.sampling.image_sampler import generate_ddpm_samples, generate_vae_samples
from src.utils.image_utils import denorm


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", choices=["vae", "ddpm", "both"], default="both")
    parser.add_argument("--n", type=int, default=16)
    args = parser.parse_args()

    cfg = Config()
    evaluator = GenAIEvaluator(cfg)
    os.makedirs(cfg.samples_dir, exist_ok=True)

    if args.model in ("vae", "both"):
        vae = evaluator._load_vae()
        imgs = generate_vae_samples(vae, args.n, evaluator.device)
        for i, img in enumerate(imgs):
            save_image(denorm(img), os.path.join(cfg.samples_dir, f"vae_{i:03d}.png"))
        print(f"Saved {args.n} VAE samples to {cfg.samples_dir}")

    if args.model in ("ddpm", "both"):
        unet, diffusion, ema = evaluator._load_ddpm()
        eval_unet = UNet.from_config(cfg).to(evaluator.device)
        ema.copy_to(eval_unet)
        imgs = generate_ddpm_samples(eval_unet, diffusion, cfg, args.n, evaluator.device)
        for i, img in enumerate(imgs):
            save_image(denorm(img), os.path.join(cfg.samples_dir, f"ddpm_{i:03d}.png"))
        print(f"Saved {args.n} DDPM samples to {cfg.samples_dir}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/generate_comparison_chart.py
"""
Generates outputs/comparison/version_comparison.png — a two-panel chart
comparing FID and Inception Score across all training versions. Re-run this
whenever a new version's results.json is added, after updating VERSIONS below.
"""
import matplotlib.pyplot as plt

# ---- Data: update this list as new versions are trained -------------------
VERSIONS = ["v1.0", "v1.1", "v1.2"]
VAE_FID = [120.69, 97.94, 97.06]
DDPM_FID = [43.77, 28.37, 26.42]
VAE_IS = [2.04, 2.00, 2.00]
DDPM_IS = [2.79, 2.59, 2.61]
NOTES = [
    "linear schedule, no x0 clip,\n30k images, 30/40 epochs",
    "x0-clip fix, linear schedule,\n40k images, 40/50 epochs",
    "cosine schedule, extra attention,\ngrad clip, LR decay",
]

VAE_COLOR = "#4C72B0"
DDPM_COLOR = "#DD8452"

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
x = range(len(VERSIONS))

# ---- Panel 1: FID (the headline metric) ------------------------------------
ax = axes[0]
ax.plot(x, VAE_FID, marker="o", markersize=7, linewidth=2, color=VAE_COLOR, label="VAE")
ax.plot(x, DDPM_FID, marker="o", markersize=7, linewidth=2, color=DDPM_COLOR, label="DDPM")

for i, v in enumerate(VAE_FID):
    ax.annotate(f"{v:.1f}", (i, v), textcoords="offset points", xytext=(0, 10),
                ha="center", fontsize=9, color=VAE_COLOR, fontweight="bold")
for i, v in enumerate(DDPM_FID):
    ax.annotate(f"{v:.1f}", (i, v), textcoords="offset points", xytext=(0, -16),
                ha="center", fontsize=9, color=DDPM_COLOR, fontweight="bold")

# percentage-change annotations between consecutive points (DDPM = the headline story)
for i in range(len(DDPM_FID) - 1):
    pct = (DDPM_FID[i + 1] - DDPM_FID[i]) / DDPM_FID[i] * 100
    mid_x = i + 0.5
    mid_y = (DDPM_FID[i] + DDPM_FID[i + 1]) / 2 + 4
    ax.annotate(f"{pct:+.1f}%", (mid_x, mid_y), ha="center", fontsize=8.5,
                color=DDPM_COLOR, style="italic")

ax.set_xticks(list(x))
ax.set_xticklabels(VERSIONS, fontsize=11)
ax.set_ylabel("FID (lower is better)", fontsize=11)
ax.set_title("Frechet Inception Distance across versions", fontsize=12, fontweight="bold")
ax.legend(loc="upper right", fontsize=10, frameon=True)
ax.grid(True, alpha=0.25, linestyle="--")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_ylim(0, max(VAE_FID) * 1.2)

# ---- Panel 2: Inception Score (secondary metric, noted as weaker signal) --
ax2 = axes[1]
ax2.plot(x, VAE_IS, marker="o", markersize=7, linewidth=2, color=VAE_COLOR, label="VAE")
ax2.plot(x, DDPM_IS, marker="o", markersize=7, linewidth=2, color=DDPM_COLOR, label="DDPM")

for i, v in enumerate(VAE_IS):
    ax2.annotate(f"{v:.2f}", (i, v), textcoords="offset points", xytext=(0, 10),
                 ha="center", fontsize=9, color=VAE_COLOR, fontweight="bold")
for i, v in enumerate(DDPM_IS):
    ax2.annotate(f"{v:.2f}", (i, v), textcoords="offset points", xytext=(0, -16),
                 ha="center", fontsize=9, color=DDPM_COLOR, fontweight="bold")

ax2.set_xticks(list(x))
ax2.set_xticklabels(VERSIONS, fontsize=11)
ax2.set_ylabel("Inception Score (higher is better)", fontsize=11)
ax2.set_title("Inception Score across versions", fontsize=12, fontweight="bold")
ax2.legend(loc="lower right", fontsize=10, frameon=True)
ax2.grid(True, alpha=0.25, linestyle="--")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.set_ylim(1.5, 3.2)

fig.suptitle("VAE vs DDPM — quantitative comparison across training versions",
             fontsize=13, fontweight="bold", y=1.02)

# Footnote with what changed at each version, small and unobtrusive
footnote = "  |  ".join(f"{v}: {n.replace(chr(10), ' ')}" for v, n in zip(VERSIONS, NOTES))
fig.text(0.5, -0.06, footnote, ha="center", fontsize=7.5, color="#555555", wrap=True)

plt.tight_layout()
plt.savefig("/home/claude/Gen_AI/outputs/comparison/version_comparison.png",
            dpi=200, bbox_inches="tight")
print("Saved outputs/comparison/version_comparison.png")

In [ ]:
%%writefile src/__init__.py

In [ ]:
%%writefile src/datasets/__init__.py

In [ ]:
%%writefile src/datasets/augmentations.py
"""Light data augmentation, kept separate from the core resize/normalize
transform so it can be toggled independently (e.g. disabled during
evaluation/sampling, always disabled for reconstruction visualizations)."""
from torchvision import transforms


def build_augmentations(horizontal_flip: bool = True):
    aug = []
    if horizontal_flip:
        aug.append(transforms.RandomHorizontalFlip(p=0.5))
    return aug

In [ ]:
%%writefile src/datasets/dataloader.py
"""Builds the CelebA DataLoader from a Config."""
import torch
from torch.utils.data import DataLoader

from src.datasets.dataset import CelebADataset
from src.datasets.preprocessing import list_image_files
from src.datasets.transforms import build_transform


def get_dataloader(cfg, augment: bool = True):
    """Returns (dataloader, list_of_file_paths, eval_transform).

    `eval_transform` has no augmentation applied — use it for evaluation /
    reconstruction / sampling comparisons where you want deterministic
    preprocessing of real images.
    """
    files = list_image_files(cfg.dataset.data_dir, cfg.dataset.subset_size, seed=cfg.seed)

    train_transform = build_transform(cfg.dataset.img_size, augment=augment,
                                       horizontal_flip=cfg.dataset.horizontal_flip)
    eval_transform = build_transform(cfg.dataset.img_size, augment=False)

    dataset = CelebADataset(files, train_transform)
    loader = DataLoader(
        dataset,
        batch_size=cfg.dataset.batch_size,
        shuffle=True,
        num_workers=cfg.dataset.num_workers,
        drop_last=True,
        pin_memory=torch.cuda.is_available(),
    )
    return loader, files, eval_transform

In [ ]:
%%writefile src/datasets/dataset.py
"""CelebA torch Dataset."""
from PIL import Image
from torch.utils.data import Dataset


class CelebADataset(Dataset):
    def __init__(self, file_list, transform):
        self.files = file_list
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        return self.transform(img)

In [ ]:
%%writefile src/datasets/preprocessing.py
"""Locating and listing the CelebA image files on disk (Kaggle or local)."""
import glob
import os
import random

CELEBA_DIR_CANDIDATES = [
    "/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba",
    "/kaggle/input/celeba-dataset/img_align_celeba/img_align_celeba",
    "/kaggle/input/celeba-dataset/img_align_celeba",
]


def resolve_celeba_dir(explicit_dir: str = "") -> str:
    candidates = [explicit_dir] if explicit_dir else CELEBA_DIR_CANDIDATES
    for c in candidates:
        if c and os.path.isdir(c) and len(os.listdir(c)) > 0:
            return c
    raise FileNotFoundError(
        "CelebA image folder not found in any known location. Add the "
        "'CelebFaces Attributes (CelebA) Dataset' to this Kaggle notebook, or "
        "set cfg.dataset.data_dir explicitly to the correct path."
    )


def list_image_files(data_dir: str, subset_size: int, seed: int = 42):
    img_dir = resolve_celeba_dir(data_dir)
    all_files = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
    if not all_files:
        raise FileNotFoundError(f"No .jpg files found in {img_dir}")

    rng = random.Random(seed)
    rng.shuffle(all_files)
    return all_files[:subset_size]

In [ ]:
%%writefile src/datasets/transforms.py
"""Image preprocessing transform pipeline."""
from torchvision import transforms

from src.datasets.augmentations import build_augmentations


def build_transform(img_size: int, augment: bool = False, horizontal_flip: bool = True) -> transforms.Compose:
    ops = [transforms.CenterCrop(178)]  # CelebA faces are roughly centered in a 178x178 box
    if augment:
        ops += build_augmentations(horizontal_flip=horizontal_flip)
    ops += [
        transforms.Resize(img_size),
        transforms.ToTensor(),
        transforms.Normalize([0.5] * 3, [0.5] * 3),  # -> [-1, 1]
    ]
    return transforms.Compose(ops)

In [ ]:
%%writefile src/engine/__init__.py

In [ ]:
%%writefile src/engine/evaluator.py
"""
Thin orchestration layer for evaluation: loads checkpoints if models aren't
already in memory, then delegates all metric computation to
src/evaluation/evaluator.py. This is the entry point evaluate.py calls.
"""
import os

from src.datasets.dataloader import get_dataloader
from src.evaluation.evaluator import Evaluator
from src.models.diffusion import GaussianDiffusion
from src.models.unet import UNet
from src.models.vae import VAE
from src.training.ema import EMA
from src.utils.checkpoint import load_checkpoint
from src.utils.device import get_device


class GenAIEvaluator:
    def __init__(self, cfg):
        self.cfg = cfg
        self.device = get_device()

    def _load_vae(self):
        vae = VAE.from_config(self.cfg).to(self.device)
        ckpt = load_checkpoint(os.path.join(self.cfg.checkpoint_dir, "vae_best.pt"),
                                map_location=self.device)
        vae.load_state_dict(ckpt["model"])
        return vae

    def _load_ddpm(self):
        unet = UNet.from_config(self.cfg).to(self.device)
        ckpt = load_checkpoint(os.path.join(self.cfg.checkpoint_dir, "ddpm_best.pt"),
                                map_location=self.device)
        ema = EMA(unet, decay=self.cfg.diffusion.ema_decay)
        ema.load_state_dict(ckpt["ema"])
        diffusion = GaussianDiffusion.from_config(self.cfg, self.device)
        return unet, diffusion, ema

    def run(self, vae=None, unet=None, diffusion=None, ema=None, files=None, transform=None):
        if files is None or transform is None:
            _, files, transform = get_dataloader(self.cfg)
        if vae is None:
            vae = self._load_vae()
        if unet is None or diffusion is None or ema is None:
            unet, diffusion, ema = self._load_ddpm()

        evaluator = Evaluator(self.cfg, self.device)
        return evaluator.evaluate(vae, unet, diffusion, ema, files, transform)

In [ ]:
%%writefile src/engine/inferencer.py
"""
Thin orchestration layer for single-image / single-sample inference: loads
checkpoints and exposes simple encode/decode/generate calls. Used by
inference.py.
"""
import os

import torch
from PIL import Image

from src.datasets.transforms import build_transform
from src.models.diffusion import GaussianDiffusion
from src.models.unet import UNet
from src.models.vae import VAE
from src.sampling.ddim_sampler import ddim_sample
from src.sampling.ddpm_sampler import ddpm_sample
from src.training.ema import EMA
from src.utils.checkpoint import load_checkpoint
from src.utils.device import get_device
from src.utils.image_utils import denorm


class GenAIInferencer:
    def __init__(self, cfg):
        self.cfg = cfg
        self.device = get_device()
        self.transform = build_transform(cfg.dataset.img_size, augment=False)

        self.vae = VAE.from_config(cfg).to(self.device)
        vae_ckpt = load_checkpoint(os.path.join(cfg.checkpoint_dir, "vae_best.pt"),
                                    map_location=self.device)
        self.vae.load_state_dict(vae_ckpt["model"])
        self.vae.eval()

        self.unet = UNet.from_config(cfg).to(self.device)
        ddpm_ckpt = load_checkpoint(os.path.join(cfg.checkpoint_dir, "ddpm_best.pt"),
                                     map_location=self.device)
        self.ema = EMA(self.unet, decay=cfg.diffusion.ema_decay)
        self.ema.load_state_dict(ddpm_ckpt["ema"])
        self.ema.copy_to(self.unet)
        self.unet.eval()
        self.diffusion = GaussianDiffusion.from_config(cfg, self.device)

    @torch.no_grad()
    def reconstruct_image(self, image_path: str):
        """VAE encode -> decode a single real image, returns (input, reconstruction) as [0,1] tensors."""
        img = Image.open(image_path).convert("RGB")
        x = self.transform(img).unsqueeze(0).to(self.device)
        recon, _, _ = self.vae(x)
        return denorm(x[0]).cpu(), denorm(recon[0]).cpu()

    @torch.no_grad()
    def generate_one(self, source: str = "ddpm"):
        """Generates a single new image from either model ("vae" or "ddpm")."""
        if source == "vae":
            img = self.vae.sample(1, self.device)[0]
        elif self.cfg.diffusion.sampler == "ddim":
            img = ddim_sample(self.unet, self.diffusion, self.cfg.dataset.img_size, 1,
                               device=self.device, ddim_steps=self.cfg.diffusion.ddim_steps,
                               eta=self.cfg.diffusion.ddim_eta)[0]
        else:
            img = ddpm_sample(self.unet, self.diffusion, self.cfg.dataset.img_size, 1,
                               device=self.device)[0]
        return denorm(img).cpu()

In [ ]:
%%writefile src/engine/trainer.py
"""
Thin orchestration layer: runs the VAE trainer then the DDPM trainer using
one shared dataloader. Contains no training logic itself — that lives in
src/training/train_vae.py and src/training/train_ddpm.py. This is the single
entry point train.py calls.
"""
from src.datasets.dataloader import get_dataloader
from src.training.train_ddpm import train_ddpm
from src.training.train_vae import train_vae
from src.utils.paths import ensure_output_dirs


class GenAITrainer:
    def __init__(self, cfg):
        self.cfg = cfg
        ensure_output_dirs(cfg)
        self.loader, self.files, self.eval_transform = get_dataloader(cfg)

    def run(self):
        print("=" * 60); print("Training VAE"); print("=" * 60)
        vae, vae_history = train_vae(self.cfg, loader=self.loader)

        print("=" * 60); print("Training DDPM"); print("=" * 60)
        unet, diffusion, ema, ddpm_history = train_ddpm(self.cfg, loader=self.loader)

        return {
            "vae": vae, "vae_history": vae_history,
            "unet": unet, "diffusion": diffusion, "ema": ema, "ddpm_history": ddpm_history,
            "files": self.files, "eval_transform": self.eval_transform,
        }

In [ ]:
%%writefile src/evaluation/__init__.py

In [ ]:
%%writefile src/evaluation/evaluator.py
"""
Orchestrates the full quantitative evaluation: generates samples from both
models, computes FID / Inception Score / VAE reconstruction metrics, and
saves the results + comparison grids to disk.
"""
import json
import os

import numpy as np
import torch
from PIL import Image

from src.evaluation.fid import compute_fid
from src.evaluation.quality import compute_inception_score
from src.evaluation.reconstruction import compute_reconstruction_metrics
from src.models.unet import UNet
from src.sampling.image_sampler import generate_ddpm_samples, generate_vae_samples
from src.visualization.sampling_grid import save_comparison_grid, save_grid


class Evaluator:
    def __init__(self, cfg, device):
        self.cfg = cfg
        self.device = device

    @torch.no_grad()
    def _get_real_batch(self, files, transform, n, seed=0):
        rng = np.random.RandomState(seed)
        idx = rng.choice(len(files), size=min(n, len(files)), replace=False)
        imgs = [transform(Image.open(files[i]).convert("RGB")) for i in idx]
        return torch.stack(imgs)

    def evaluate(self, vae, unet, diffusion, ema, files, transform):
        cfg = self.cfg
        device = self.device

        eval_unet = UNet.from_config(cfg).to(device)
        ema.copy_to(eval_unet)
        eval_unet.eval()

        print("Generating real reference batch...")
        real_imgs = self._get_real_batch(files, transform, cfg.n_eval_samples, seed=cfg.seed)
        print("Generating VAE samples...")
        vae_imgs = generate_vae_samples(vae, cfg.n_eval_samples, device, batch=cfg.eval_batch_size)
        print(f"Generating DDPM samples (sampler={cfg.diffusion.sampler})...")
        ddpm_imgs = generate_ddpm_samples(eval_unet, diffusion, cfg, cfg.n_eval_samples, device,
                                           batch=cfg.eval_batch_size)

        print("Computing FID / Inception Score...")
        vae_fid = compute_fid(real_imgs, vae_imgs, device)
        vae_is_mean, vae_is_std = compute_inception_score(vae_imgs, device)
        ddpm_fid = compute_fid(real_imgs, ddpm_imgs, device)
        ddpm_is_mean, ddpm_is_std = compute_inception_score(ddpm_imgs, device)

        recon_metrics = compute_reconstruction_metrics(vae, real_imgs[:64], device)

        results = {
            "vae": {"fid": vae_fid, "is_mean": vae_is_mean, "is_std": vae_is_std, **recon_metrics},
            "ddpm": {"fid": ddpm_fid, "is_mean": ddpm_is_mean, "is_std": ddpm_is_std,
                     "sampler": cfg.diffusion.sampler},
            "n_eval_samples": cfg.n_eval_samples,
        }

        os.makedirs(cfg.output_dir, exist_ok=True)
        with open(os.path.join(cfg.output_dir, "results.json"), "w") as f:
            json.dump(results, f, indent=2)

        save_grid(real_imgs, "Real CelebA images", os.path.join(cfg.output_dir, "real_grid.png"))
        save_grid(vae_imgs, "VAE samples", os.path.join(cfg.output_dir, "vae_grid.png"))
        save_grid(ddpm_imgs, "DDPM samples", os.path.join(cfg.output_dir, "ddpm_grid.png"))
        save_comparison_grid(real_imgs, vae_imgs, ddpm_imgs,
                              os.path.join(cfg.output_dir, "comparison_grid.png"))

        return results, real_imgs, vae_imgs, ddpm_imgs

In [ ]:
%%writefile src/evaluation/fid.py
"""Frechet Inception Distance, via torchmetrics/torch-fidelity's InceptionV3
feature extractor (same implementation used in most published papers)."""
from src.utils.image_utils import to_uint8


def compute_fid(real, fake, device, batch=64):
    from torchmetrics.image.fid import FrechetInceptionDistance

    fid = FrechetInceptionDistance(feature=2048, normalize=False).to(device)
    for i in range(0, len(real), batch):
        fid.update(to_uint8(real[i:i + batch]).to(device), real=True)
    for i in range(0, len(fake), batch):
        fid.update(to_uint8(fake[i:i + batch]).to(device), real=False)
    return fid.compute().item()

In [ ]:
%%writefile src/evaluation/quality.py
"""Inception Score. Noted limitation: IS is derived from an ImageNet-1000-class
classifier and is less informative for a single-class domain like faces —
report it, but weight FID more heavily in conclusions (see README)."""
from src.utils.image_utils import to_uint8


def compute_inception_score(fake, device, batch=64):
    from torchmetrics.image.inception import InceptionScore

    inc = InceptionScore(normalize=False).to(device)
    for i in range(0, len(fake), batch):
        inc.update(to_uint8(fake[i:i + batch]).to(device))
    is_mean, is_std = inc.compute()
    return is_mean.item(), is_std.item()

In [ ]:
%%writefile src/evaluation/reconstruction.py
"""VAE-specific reconstruction-quality metrics (only meaningful for the VAE,
since DDPM has no encoder / explicit reconstruction step)."""
import torch
import torch.nn.functional as F


@torch.no_grad()
def compute_reconstruction_metrics(vae, real_batch, device):
    vae.eval()
    x = real_batch.to(device)
    recon, mu, logvar = vae(x)

    mse = F.mse_loss(recon, x).item()
    # PSNR assuming pixel range [-1, 1] -> data range 2.0
    psnr = 10 * torch.log10(torch.tensor(2.0 ** 2) / max(mse, 1e-10)).item()

    return {"reconstruction_mse": mse, "psnr_db": psnr}

In [ ]:
%%writefile src/losses/__init__.py

In [ ]:
%%writefile src/losses/diffusion_loss.py
"""DDPM training loss: simple noise-prediction MSE (L_simple from Ho et al. 2020)."""
import torch.nn.functional as F


def simple_diffusion_loss(pred_noise, true_noise):
    return F.mse_loss(pred_noise, true_noise)

In [ ]:
%%writefile src/losses/perceptual_loss.py
"""
Optional VGG-based perceptual loss for the VAE. Comparing reconstructions in
a pretrained feature space (rather than raw pixels) is a well-known way to
reduce VAE blur. Off by default (configs/training.py: use_perceptual_loss)
since it adds a ~15M-parameter frozen VGG16 to memory/compute.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import vgg16, VGG16_Weights


class PerceptualLoss(nn.Module):
    def __init__(self, layer_idx: int = 16, device="cpu"):
        """layer_idx=16 corresponds to roughly the relu3_3 feature map of VGG16,
        a common choice for perceptual losses (mid-level texture/structure)."""
        super().__init__()
        vgg = vgg16(weights=VGG16_Weights.DEFAULT).features[: layer_idx + 1]
        self.vgg = vgg.to(device).eval()
        for p in self.vgg.parameters():
            p.requires_grad_(False)
        self.register_buffer(
            "mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )
        self.register_buffer(
            "std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

    def _preprocess(self, x):
        # x is in [-1, 1] -> [0, 1] -> ImageNet-normalized, as VGG expects.
        x = (x.clamp(-1, 1) + 1) / 2
        return (x - self.mean.to(x.device)) / self.std.to(x.device)

    def forward(self, recon, target):
        f_recon = self.vgg(self._preprocess(recon))
        f_target = self.vgg(self._preprocess(target))
        return F.mse_loss(f_recon, f_target)

In [ ]:
%%writefile src/losses/vae_loss.py
"""VAE objective: reconstruction + KL divergence (+ optional perceptual term)."""
import torch
import torch.nn.functional as F


def vae_loss(recon, x, mu, logvar, kld_weight=1.0, perceptual_loss_fn=None, perceptual_weight=0.0):
    """Returns (total_loss, recon_loss, kld, perceptual_loss_value).
    All terms are averaged per-sample. Pixels are assumed to be in [-1, 1]."""
    recon_loss = F.mse_loss(recon, x, reduction="sum") / x.size(0)
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    total = recon_loss + kld_weight * kld

    perceptual_value = torch.tensor(0.0, device=x.device)
    if perceptual_loss_fn is not None and perceptual_weight > 0:
        perceptual_value = perceptual_loss_fn(recon, x)
        total = total + perceptual_weight * perceptual_value

    return total, recon_loss, kld, perceptual_value

In [ ]:
%%writefile src/models/__init__.py

In [ ]:
%%writefile src/models/attention.py
"""Self-attention block used inside the U-Net at low spatial resolution."""
import torch.nn as nn


class SelfAttention(nn.Module):
    def __init__(self, ch, num_heads=4):
        super().__init__()
        self.norm = nn.GroupNorm(8, ch)
        self.mha = nn.MultiheadAttention(ch, num_heads, batch_first=True)

    def forward(self, x):
        b, c, h, w = x.shape
        h_ = self.norm(x).reshape(b, c, h * w).transpose(1, 2)  # (b, hw, c)
        out, _ = self.mha(h_, h_, h_)
        out = out.transpose(1, 2).reshape(b, c, h, w)
        return x + out

In [ ]:
%%writefile src/models/decoder.py
"""VAE convolutional decoder."""
import torch
import torch.nn as nn
import torch.nn.functional as F


class DeconvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=2, activation="relu"):
        super().__init__()
        self.deconv = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=stride, padding=1)
        self.bn = nn.BatchNorm2d(out_ch)
        self.activation = activation

    def forward(self, x):
        x = self.deconv(x)
        if self.activation == "relu":
            x = F.relu(self.bn(x), inplace=True)
        elif self.activation == "tanh":
            x = torch.tanh(x)
        return x


class Decoder(nn.Module):
    """latent_dim -> img_size x img_size x img_channels."""

    def __init__(self, img_channels=3, base_ch=64, latent_dim=128, img_size=64):
        super().__init__()
        n_up = int(torch.log2(torch.tensor(img_size // 4)).item())
        chs = [base_ch * (2 ** i) for i in range(n_up)][::-1]  # e.g. [512,256,128,64]
        self.final_ch = chs[0]
        self.spatial = 4
        flat_dim = self.final_ch * self.spatial * self.spatial
        self.fc = nn.Linear(latent_dim, flat_dim)

        chs = chs + [img_channels]
        layers = [DeconvBlock(chs[i], chs[i + 1], activation="relu") for i in range(n_up - 1)]
        layers.append(DeconvBlock(chs[-2], chs[-1], activation="tanh"))
        self.deconv = nn.Sequential(*layers)

    def forward(self, z):
        h = self.fc(z).view(-1, self.final_ch, self.spatial, self.spatial)
        return self.deconv(h)

In [ ]:
%%writefile src/models/diffusion.py
"""
Gaussian diffusion process (Ho et al. 2020): precomputed beta/alpha schedules,
the forward (noising) process q_sample, and the training loss. Reverse
sampling lives in src/sampling/ (ddpm_sampler.py, ddim_sampler.py) since
sampling strategy is a separate concern from the noise schedule itself.
"""
import torch
import torch.nn.functional as F


class GaussianDiffusion:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cpu",
                 beta_schedule="cosine"):
        self.T = timesteps
        self.device = device
        self.betas = self._make_beta_schedule(beta_schedule, timesteps, beta_start, beta_end, device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        self.posterior_variance = (
            self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
        # Coefficients for computing the posterior mean from a (clipped) x0
        # estimate rather than directly from epsilon — see p_sample in
        # src/sampling/ddpm_sampler.py. Standard fix for the numerical drift
        # that otherwise produces solid black/white patches in some samples.
        self.posterior_mean_coef1 = (
            self.betas * torch.sqrt(self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
        self.posterior_mean_coef2 = (
            (1.0 - self.alphas_cumprod_prev) * torch.sqrt(self.alphas) / (1.0 - self.alphas_cumprod)
        )

    @staticmethod
    def _make_beta_schedule(schedule, timesteps, beta_start, beta_end, device):
        if schedule == "linear":
            return torch.linspace(beta_start, beta_end, timesteps, device=device)
        elif schedule == "cosine":
            # Nichol & Dhariwal, "Improved Denoising Diffusion Probabilistic
            # Models" (2021), eq. 17. s=0.008 offset avoids beta_t being too
            # small near t=0. Betas are derived from a cosine alpha_bar curve
            # and clipped to 0.999 to avoid numerical issues near t=T.
            s = 0.008
            steps = timesteps + 1
            t = torch.linspace(0, timesteps, steps, device=device) / timesteps
            alphas_cumprod = torch.cos((t + s) / (1 + s) * torch.pi * 0.5) ** 2
            alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
            betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
            return betas.clamp(max=0.999)
        else:
            raise ValueError(f"Unknown beta_schedule: {schedule!r} (expected 'linear' or 'cosine')")

    def extract(self, arr, t, shape):
        out = arr.gather(0, t)
        return out.reshape(t.shape[0], *((1,) * (len(shape) - 1)))

    def q_sample(self, x0, t, noise=None):
        """Forward process: sample x_t given x_0 and timestep t."""
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ac = self.extract(self.sqrt_alphas_cumprod, t, x0.shape)
        sqrt_omac = self.extract(self.sqrt_one_minus_alphas_cumprod, t, x0.shape)
        return sqrt_ac * x0 + sqrt_omac * noise, noise

    def training_loss(self, model, x0, loss_fn=None):
        """Noise-prediction loss. `loss_fn` defaults to simple MSE
        (see src/losses/diffusion_loss.py) but can be swapped."""
        from src.losses.diffusion_loss import simple_diffusion_loss
        loss_fn = loss_fn or simple_diffusion_loss

        b = x0.size(0)
        t = torch.randint(0, self.T, (b,), device=x0.device).long()
        x_t, noise = self.q_sample(x0, t)
        pred_noise = model(x_t, t)
        return loss_fn(pred_noise, noise)

    @classmethod
    def from_config(cls, cfg, device):
        return cls(
            timesteps=cfg.diffusion.timesteps,
            beta_start=cfg.diffusion.beta_start,
            beta_end=cfg.diffusion.beta_end,
            device=device,
            beta_schedule=cfg.diffusion.beta_schedule,
        )

In [ ]:
%%writefile src/models/encoder.py
"""VAE convolutional encoder."""
import torch
import torch.nn as nn


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=2):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=stride, padding=1)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class Encoder(nn.Module):
    """img_size x img_size x img_channels -> latent_dim (mu, logvar)."""

    def __init__(self, img_channels=3, base_ch=64, latent_dim=128, img_size=64):
        super().__init__()
        n_down = int(torch.log2(torch.tensor(img_size // 4)).item())  # downsample until 4x4
        chs = [img_channels] + [base_ch * (2 ** i) for i in range(n_down)]
        layers = [ConvBlock(chs[i], chs[i + 1]) for i in range(n_down)]
        self.conv = nn.Sequential(*layers)
        self.final_ch = chs[-1]
        self.spatial = 4
        flat_dim = self.final_ch * self.spatial * self.spatial
        self.fc_mu = nn.Linear(flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(flat_dim, latent_dim)

    def forward(self, x):
        h = self.conv(x).flatten(1)
        mu = self.fc_mu(h)
        # Clamp logvar for numerical stability (prevents the KL blow-up that
        # can happen in the first epoch before KL-annealing kicks in).
        logvar = torch.clamp(self.fc_logvar(h), -10.0, 10.0)
        return mu, logvar

In [ ]:
%%writefile src/models/unet.py
"""
U-Net noise-prediction network for DDPM: sinusoidal timestep embeddings,
residual blocks conditioned on time, self-attention at the bottleneck plus
(optionally) additional resolutions (src/models/attention.py), and
strided-conv down/up-sampling.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from src.models.attention import SelfAttention


class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
        args = t[:, None].float() * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Linear(time_dim, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.act = nn.SiLU()

    def forward(self, x, t_emb):
        h = self.conv1(self.act(self.norm1(x)))
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        h = self.conv2(self.act(self.norm2(h)))
        return h + self.skip(x)


class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.Conv2d(ch, ch, 3, stride=2, padding=1)

    def forward(self, x):
        return self.op(x)


class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.ConvTranspose2d(ch, ch, 4, stride=2, padding=1)

    def forward(self, x):
        return self.op(x)


class UNet(nn.Module):
    """Predicts the noise epsilon added to x_t.

    Attention is always applied at the bottleneck, plus at any resolution
    listed in `attn_resolutions` (matched against the spatial size each
    down/up level operates at, computed from `img_size`). E.g. with
    img_size=64 and ch_mults=(1,2,4,4), levels operate at 64/32/16/8 px;
    attn_resolutions=(16,) adds attention at the 16px level in addition to
    the 8px bottleneck.
    """

    def __init__(self, img_channels=3, base_ch=64, ch_mults=(1, 2, 4, 4), time_dim=256,
                 attn_heads=4, img_size=64, attn_resolutions=()):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalTimeEmbedding(base_ch),
            nn.Linear(base_ch, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        self.in_conv = nn.Conv2d(img_channels, base_ch, 3, padding=1)

        chs = [base_ch * m for m in ch_mults]
        # Resolution each level's ResBlock operates at, BEFORE that level's
        # downsample is applied (level 0 = full img_size).
        level_resolutions = [img_size // (2 ** i) for i in range(len(chs))]

        self.down_blocks = nn.ModuleList()
        self.down_attn = nn.ModuleList()
        self.down_samples = nn.ModuleList()
        in_ch = base_ch
        self.skip_chs = [base_ch]
        for i, out_ch in enumerate(chs):
            self.down_blocks.append(ResBlock(in_ch, out_ch, time_dim))
            in_ch = out_ch
            self.skip_chs.append(in_ch)
            use_attn = level_resolutions[i] in attn_resolutions
            self.down_attn.append(SelfAttention(in_ch, num_heads=attn_heads) if use_attn else nn.Identity())
            self.down_samples.append(Downsample(in_ch) if i < len(chs) - 1 else nn.Identity())

        self.mid_block1 = ResBlock(in_ch, in_ch, time_dim)
        self.mid_attn = SelfAttention(in_ch, num_heads=attn_heads)
        self.mid_block2 = ResBlock(in_ch, in_ch, time_dim)

        self.up_blocks = nn.ModuleList()
        self.up_attn = nn.ModuleList()
        self.up_samples = nn.ModuleList()
        rev_chs = list(reversed(chs))
        rev_resolutions = list(reversed(level_resolutions))
        for i, out_ch in enumerate(rev_chs):
            skip_ch = self.skip_chs.pop()
            self.up_blocks.append(ResBlock(in_ch + skip_ch, out_ch, time_dim))
            in_ch = out_ch
            use_attn = rev_resolutions[i] in attn_resolutions
            self.up_attn.append(SelfAttention(in_ch, num_heads=attn_heads) if use_attn else nn.Identity())
            self.up_samples.append(Upsample(in_ch) if i < len(rev_chs) - 1 else nn.Identity())

        self.out_norm = nn.GroupNorm(8, in_ch)
        self.out_conv = nn.Conv2d(in_ch, img_channels, 3, padding=1)
        self.act = nn.SiLU()

    def forward(self, x, t):
        t_emb = self.time_embed(t)
        h = self.in_conv(x)
        skips = [h]
        for block, attn, down in zip(self.down_blocks, self.down_attn, self.down_samples):
            h = block(h, t_emb)
            h = attn(h)
            skips.append(h)
            h = down(h)

        h = self.mid_block1(h, t_emb)
        h = self.mid_attn(h)
        h = self.mid_block2(h, t_emb)

        for block, attn, up in zip(self.up_blocks, self.up_attn, self.up_samples):
            skip = skips.pop()
            if h.shape[-2:] != skip.shape[-2:]:
                h = F.interpolate(h, size=skip.shape[-2:], mode="nearest")
            h = torch.cat([h, skip], dim=1)
            h = block(h, t_emb)
            h = attn(h)
            h = up(h)

        return self.out_conv(self.act(self.out_norm(h)))

    @classmethod
    def from_config(cls, cfg):
        return cls(
            img_channels=cfg.model.img_channels,
            base_ch=cfg.model.unet_base_ch,
            ch_mults=cfg.model.unet_ch_mults,
            time_dim=cfg.model.unet_time_dim,
            attn_heads=cfg.model.unet_attn_heads,
            img_size=cfg.dataset.img_size,
            attn_resolutions=cfg.model.unet_attn_resolutions,
        )

In [ ]:
%%writefile src/models/vae.py
"""Variational Autoencoder: wires together Encoder + Decoder + reparameterization."""
import torch
import torch.nn as nn

from src.models.decoder import Decoder
from src.models.encoder import Encoder


class VAE(nn.Module):
    def __init__(self, img_channels=3, base_ch=64, latent_dim=128, img_size=64):
        super().__init__()
        self.encoder = Encoder(img_channels, base_ch, latent_dim, img_size)
        self.decoder = Decoder(img_channels, base_ch, latent_dim, img_size)
        self.latent_dim = latent_dim

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

    @torch.no_grad()
    def encode(self, x):
        """Deterministic encode (returns mu), used for reconstruction /
        latent-space visualizations rather than stochastic sampling."""
        mu, logvar = self.encoder(x)
        return mu, logvar

    @torch.no_grad()
    def decode(self, z):
        return self.decoder(z)

    @torch.no_grad()
    def sample(self, n, device):
        z = torch.randn(n, self.latent_dim, device=device)
        return self.decoder(z)

    @classmethod
    def from_config(cls, cfg):
        return cls(
            img_channels=cfg.model.img_channels,
            base_ch=cfg.model.vae_base_ch,
            latent_dim=cfg.model.latent_dim,
            img_size=cfg.dataset.img_size,
        )

In [ ]:
%%writefile src/sampling/__init__.py

In [ ]:
%%writefile src/sampling/ddim_sampler.py
"""
DDIM sampler (Song et al. 2020): a deterministic (or partially stochastic,
via `eta`), non-Markovian reverse process that reuses a DDPM-trained model
but needs far fewer steps (e.g. 50 instead of 1000) — directly addresses the
"DDPM sampling is slow" bottleneck at some sample-quality cost.
"""
import torch


@torch.no_grad()
def ddim_sample(model, diffusion, img_size, batch_size, channels=3, device="cpu",
                 ddim_steps=50, eta=0.0):
    step_indices = torch.linspace(0, diffusion.T - 1, ddim_steps, device=device).long()
    step_indices = torch.flip(step_indices, dims=[0])  # T-1 -> 0

    x = torch.randn(batch_size, channels, img_size, img_size, device=device)

    for i, t_val in enumerate(step_indices):
        t = torch.full((batch_size,), t_val.item(), device=device, dtype=torch.long)
        pred_noise = model(x, t)

        alpha_t = diffusion.extract(diffusion.alphas_cumprod, t, x.shape)
        if i < len(step_indices) - 1:
            t_prev_val = step_indices[i + 1]
            t_prev = torch.full((batch_size,), t_prev_val.item(), device=device, dtype=torch.long)
            alpha_prev = diffusion.extract(diffusion.alphas_cumprod, t_prev, x.shape)
        else:
            alpha_prev = torch.ones_like(alpha_t)

        # Predicted x0 from the current noise estimate
        pred_x0 = (x - torch.sqrt(1 - alpha_t) * pred_noise) / torch.sqrt(alpha_t)
        pred_x0 = pred_x0.clamp(-1, 1)

        sigma_t = eta * torch.sqrt((1 - alpha_prev) / (1 - alpha_t)) * torch.sqrt(1 - alpha_t / alpha_prev)
        dir_xt = torch.sqrt((1 - alpha_prev - sigma_t ** 2).clamp(min=0)) * pred_noise
        noise = torch.randn_like(x) if eta > 0 else 0.0

        x = torch.sqrt(alpha_prev) * pred_x0 + dir_xt + sigma_t * noise

    return x

In [ ]:
%%writefile src/sampling/ddpm_sampler.py
"""Full ancestral DDPM reverse sampling (T sequential steps). Slow but exactly
matches the training-time process — use this for final reported results."""
import torch


@torch.no_grad()
def ddpm_p_sample_step(model, diffusion, x_t, t, clip_denoised=True):
    """One reverse step: p(x_{t-1} | x_t).

    Predicts x0 from the model's noise estimate, optionally clips it to the
    valid [-1, 1] pixel range, then computes the posterior mean from that
    (clipped) x0. Without this clipping, small per-step numerical errors can
    compound over ~1000 steps and drive some samples to solid black/white —
    clipping the intermediate x0 estimate at every step is the standard fix
    used in the official DDPM/Improved-DDPM implementations.
    """
    sqrt_recip_alphas_cumprod_t = diffusion.extract(
        1.0 / torch.sqrt(diffusion.alphas_cumprod), t, x_t.shape
    )
    sqrt_recipm1_alphas_cumprod_t = diffusion.extract(
        torch.sqrt(1.0 / diffusion.alphas_cumprod - 1.0), t, x_t.shape
    )

    pred_noise = model(x_t, t)
    pred_x0 = sqrt_recip_alphas_cumprod_t * x_t - sqrt_recipm1_alphas_cumprod_t * pred_noise
    if clip_denoised:
        pred_x0 = pred_x0.clamp(-1.0, 1.0)

    coef1 = diffusion.extract(diffusion.posterior_mean_coef1, t, x_t.shape)
    coef2 = diffusion.extract(diffusion.posterior_mean_coef2, t, x_t.shape)
    model_mean = coef1 * pred_x0 + coef2 * x_t

    if (t == 0).all():
        return model_mean
    posterior_var_t = diffusion.extract(diffusion.posterior_variance, t, x_t.shape)
    noise = torch.randn_like(x_t)
    mask = (t > 0).float().reshape(-1, *((1,) * (len(x_t.shape) - 1)))
    return model_mean + mask * torch.sqrt(posterior_var_t) * noise


@torch.no_grad()
def ddpm_sample(model, diffusion, img_size, batch_size, channels=3, device="cpu",
                 return_trajectory=False, clip_denoised=True):
    """Full reverse process from pure noise x_T to x_0.
    If return_trajectory=True, also returns a list of intermediate x_t
    snapshots (used by src/visualization/denoising.py)."""
    x = torch.randn(batch_size, channels, img_size, img_size, device=device)
    trajectory = [x.cpu()] if return_trajectory else None

    for i in reversed(range(diffusion.T)):
        t = torch.full((batch_size,), i, device=device, dtype=torch.long)
        x = ddpm_p_sample_step(model, diffusion, x, t, clip_denoised=clip_denoised)
        if return_trajectory and i % max(1, diffusion.T // 10) == 0:
            trajectory.append(x.cpu())

    if return_trajectory:
        return x, trajectory
    return x

In [ ]:
%%writefile src/sampling/image_sampler.py
"""
High-level sampling API used by sample.py / inference.py: given trained
model artifacts, generate N new images from either model without the
caller needing to know about GaussianDiffusion / EMA / sampler choice.
"""
import torch

from src.sampling.ddim_sampler import ddim_sample
from src.sampling.ddpm_sampler import ddpm_sample
from src.sampling.latent_sampler import sample_latents


@torch.no_grad()
def generate_vae_samples(vae, n, device, batch=128):
    vae.eval()
    out = []
    for i in range(0, n, batch):
        b = min(batch, n - i)
        out.append(sample_latents(vae, b, device).cpu())
    return torch.cat(out)


@torch.no_grad()
def generate_ddpm_samples(unet, diffusion, cfg, n, device, batch=64):
    unet.eval()
    out = []
    sampler = cfg.diffusion.sampler
    for i in range(0, n, batch):
        b = min(batch, n - i)
        if sampler == "ddim":
            imgs = ddim_sample(unet, diffusion, cfg.dataset.img_size, b, device=device,
                                ddim_steps=cfg.diffusion.ddim_steps, eta=cfg.diffusion.ddim_eta)
        else:
            imgs = ddpm_sample(unet, diffusion, cfg.dataset.img_size, b, device=device)
        out.append(imgs.cpu())
    return torch.cat(out)

In [ ]:
%%writefile src/sampling/latent_sampler.py
"""VAE latent-space sampling helpers."""
import torch


@torch.no_grad()
def sample_latents(vae, n, device):
    """Draw n latent vectors from the VAE prior N(0, I) and decode them."""
    return vae.sample(n, device)


@torch.no_grad()
def interpolate_latents(vae, z_start, z_end, steps=10):
    """Linear interpolation between two latent vectors, decoded at each step.
    Used by src/visualization/interpolation.py."""
    alphas = torch.linspace(0, 1, steps, device=z_start.device)
    zs = torch.stack([(1 - a) * z_start + a * z_end for a in alphas])
    return vae.decode(zs)

In [ ]:
%%writefile src/training/__init__.py

In [ ]:
%%writefile src/training/callbacks.py
"""Lightweight training callbacks: checkpointing and periodic sample previews.
Kept as plain classes (not a framework) — call `.on_epoch_end(...)` manually
from the trainer loop.
"""
import os

from src.utils.checkpoint import save_checkpoint
from src.utils.image_utils import to_grid
from src.visualization.training_curves import save_curve_plot


class CheckpointCallback:
    """Saves a resumable checkpoint every `every_n` epochs, and always on the
    final epoch, plus a lightweight final-weights-only file."""

    def __init__(self, ckpt_path: str, final_path: str, every_n: int = 2):
        self.ckpt_path = ckpt_path
        self.final_path = final_path
        self.every_n = every_n

    def maybe_save(self, epoch: int, total_epochs: int, state: dict):
        if (epoch + 1) % self.every_n == 0 or epoch == total_epochs - 1:
            save_checkpoint(self.ckpt_path, epoch=epoch, **state)
            return True
        return False

    def save_final(self, **state):
        save_checkpoint(self.final_path, **state)


class SampleGridCallback:
    """Saves a sample grid image every `every_n` epochs (in addition to any
    inline plt.show() the trainer does)."""

    def __init__(self, out_dir: str, prefix: str, every_n: int = 5):
        self.out_dir = out_dir
        self.prefix = prefix
        self.every_n = every_n
        os.makedirs(out_dir, exist_ok=True)

    def maybe_save(self, epoch: int, total_epochs: int, samples):
        import matplotlib.pyplot as plt

        if (epoch + 1) % self.every_n == 0 or epoch == total_epochs - 1:
            grid = to_grid(samples, nrow=4)
            path = os.path.join(self.out_dir, f"{self.prefix}_epoch{epoch + 1:03d}.png")
            plt.figure(figsize=(5, 5))
            plt.axis("off")
            plt.imshow(grid.cpu().permute(1, 2, 0).numpy())
            plt.savefig(path, bbox_inches="tight")
            plt.close()
            return path
        return None

In [ ]:
%%writefile src/training/ema.py
"""Exponential Moving Average of model weights — standard practice for DDPM
training, noticeably stabilizes / sharpens sample quality at inference time."""
import torch


class EMA:
    def __init__(self, model: torch.nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.clone().detach() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: torch.nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.clone()

    def copy_to(self, model: torch.nn.Module):
        model.load_state_dict(self.shadow, strict=True)

    def state_dict(self):
        return self.shadow

    def load_state_dict(self, state_dict):
        self.shadow = {k: v.clone() for k, v in state_dict.items()}

In [ ]:
%%writefile src/training/optimizer.py
"""Optimizer construction, centralized so trainers don't hardcode Adam vs AdamW."""
import torch


def build_vae_optimizer(model, cfg):
    return torch.optim.Adam(model.parameters(), lr=cfg.training.vae_lr)


def build_ddpm_optimizer(model, cfg):
    return torch.optim.AdamW(model.parameters(), lr=cfg.training.ddpm_lr)

In [ ]:
%%writefile src/training/scheduler.py
"""Optional learning-rate schedulers. Disabled (constant LR) by default —
the VAE/DDPM configs above train fine with a fixed LR, but a cosine decay is
provided since Kaggle GPU-time-limited runs sometimes benefit from LR decay
in the last few epochs."""
import torch


def build_cosine_scheduler(optimizer, total_epochs: int, enabled: bool = False):
    if not enabled:
        return None
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)

In [ ]:
%%writefile src/training/train_ddpm.py
"""DDPM trainer: implements BaseTrainer for the U-Net + GaussianDiffusion."""
import os

import torch

from src.datasets.dataloader import get_dataloader
from src.models.diffusion import GaussianDiffusion
from src.models.unet import UNet
from src.sampling.ddpm_sampler import ddpm_sample
from src.training.callbacks import CheckpointCallback, SampleGridCallback
from src.training.ema import EMA
from src.training.optimizer import build_ddpm_optimizer
from src.training.scheduler import build_cosine_scheduler
from src.training.trainer import BaseTrainer
from src.utils.checkpoint import save_checkpoint
from src.utils.device import get_device
from src.utils.helpers import count_parameters
from src.utils.seed import set_seed


class DDPMTrainer(BaseTrainer):
    name = "ddpm"

    def __init__(self, cfg, loader=None):
        super().__init__(cfg, total_epochs=cfg.training.ddpm_epochs)
        set_seed(cfg.seed)
        self.device = get_device()
        self.loader = loader if loader is not None else get_dataloader(cfg)[0]

        self.model = UNet.from_config(cfg).to(self.device)
        self.logger.info(f"UNet parameters: {count_parameters(self.model):.2f}M")
        self.diffusion = GaussianDiffusion.from_config(cfg, self.device)

        self.ema = EMA(self.model, decay=cfg.diffusion.ema_decay)
        self.opt = build_ddpm_optimizer(self.model, cfg)
        self.use_amp = cfg.training.use_amp and self.device.type == "cuda"
        self.scaler = torch.amp.GradScaler("cuda", enabled=self.use_amp)
        self.grad_clip_norm = cfg.training.ddpm_grad_clip_norm
        self.scheduler = build_cosine_scheduler(
            self.opt, total_epochs=cfg.training.ddpm_epochs,
            enabled=cfg.training.ddpm_use_lr_scheduler,
        )

        self.ckpt_cb = CheckpointCallback(self.ckpt_path(), self.final_path(),
                                           every_n=cfg.training.checkpoint_every)
        self.sample_cb = SampleGridCallback(os.path.join(cfg.samples_dir, "ddpm"), "ddpm")

        self.try_resume(self.device)

        # If resuming, fast-forward the scheduler to the correct point on
        # the CURRENT cosine curve (recomputed from cfg.training.ddpm_epochs,
        # which may differ from the total used in the original run — e.g. if
        # you bumped ddpm_epochs and resumed to train longer). We deliberately
        # don't save/load the scheduler's own state_dict, since that would
        # carry over the old T_max and produce an incorrect, non-monotonic
        # LR curve after a total_epochs change.
        if self.scheduler is not None and self.start_epoch > 0:
            # CosineAnnealingLR computes each .step() recursively from the
            # CURRENT optimizer LR, not purely from a fixed base value. The
            # resumed optimizer's LR was just loaded from checkpoint already
            # decayed (e.g. near 0 at the end of the previous run) — fast-
            # forwarding from that stale value would keep it near 0 forever.
            # Reset to the true base LR first so the recursive steps recompute
            # the correct curve.
            for group in self.opt.param_groups:
                group["lr"] = cfg.training.ddpm_lr
            import warnings
            with warnings.catch_warnings():
                # Expected/benign here: we're intentionally fast-forwarding
                # the scheduler without calling optimizer.step() in between.
                warnings.filterwarnings("ignore", message="Detected call of .lr_scheduler.step.*")
                for _ in range(self.start_epoch):
                    self.scheduler.step()

    def train_one_epoch(self, epoch):
        self.model.train()
        running = 0.0
        for x in self.loader:
            x = x.to(self.device)
            self.opt.zero_grad()
            with torch.amp.autocast("cuda", enabled=self.use_amp):
                loss = self.diffusion.training_loss(self.model, x)
            self.scaler.scale(loss).backward()

            if self.grad_clip_norm and self.grad_clip_norm > 0:
                # Gradients must be unscaled before clipping, or the clip
                # threshold is meaningless relative to the AMP loss scale.
                self.scaler.unscale_(self.opt)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip_norm)

            self.scaler.step(self.opt)
            self.scaler.update()
            self.ema.update(self.model)
            running += loss.item()

        if self.scheduler is not None:
            self.scheduler.step()

        return {"loss": running / len(self.loader), "lr": self.opt.param_groups[0]["lr"]}

    def on_epoch_end(self, epoch, metrics):
        self.logger.info(
            f"epoch {epoch + 1}/{self.total_epochs}  loss={metrics['loss']:.4f}  lr={metrics['lr']:.2e}"
        )

        if (epoch + 1) % 10 == 0 or epoch == self.total_epochs - 1:
            eval_unet = UNet.from_config(self.cfg).to(self.device)
            self.ema.copy_to(eval_unet)
            eval_unet.eval()
            samples = ddpm_sample(eval_unet, self.diffusion, self.cfg.dataset.img_size,
                                   batch_size=16, device=self.device)
            self.sample_cb.maybe_save(epoch, self.total_epochs, samples)

        self.ckpt_cb.maybe_save(epoch, self.total_epochs, {**self.state_dict(), "history": self.history})

    def state_dict(self):
        return {
            "model": self.model.state_dict(),
            "optimizer": self.opt.state_dict(),
            "scaler": self.scaler.state_dict(),
            "ema": self.ema.state_dict(),
        }

    def load_state_dict(self, ckpt):
        self.model.load_state_dict(ckpt["model"])
        self.opt.load_state_dict(ckpt["optimizer"])
        self.scaler.load_state_dict(ckpt["scaler"])
        self.ema.load_state_dict(ckpt["ema"])

    def save_final(self):
        save_checkpoint(self.final_path(), model=self.model.state_dict(),
                         ema=self.ema.state_dict(), config=str(self.cfg))


def train_ddpm(cfg, loader=None):
    trainer = DDPMTrainer(cfg, loader=loader)
    history = trainer.run()
    return trainer.model, trainer.diffusion, trainer.ema, history

In [ ]:
%%writefile src/training/train_vae.py
"""VAE trainer: implements BaseTrainer for the VAE model."""
import os

import torch

from src.datasets.dataloader import get_dataloader
from src.losses.vae_loss import vae_loss
from src.models.vae import VAE
from src.training.callbacks import CheckpointCallback, SampleGridCallback
from src.training.optimizer import build_vae_optimizer
from src.training.trainer import BaseTrainer
from src.utils.checkpoint import save_checkpoint
from src.utils.device import get_device
from src.utils.helpers import count_parameters, kl_weight_schedule
from src.utils.seed import set_seed


class VAETrainer(BaseTrainer):
    name = "vae"

    def __init__(self, cfg, loader=None, perceptual_loss_fn=None):
        super().__init__(cfg, total_epochs=cfg.training.vae_epochs)
        set_seed(cfg.seed)
        self.device = get_device()
        self.loader = loader if loader is not None else get_dataloader(cfg)[0]

        self.model = VAE.from_config(cfg).to(self.device)
        self.logger.info(f"VAE parameters: {count_parameters(self.model):.2f}M")
        self.opt = build_vae_optimizer(self.model, cfg)
        self.perceptual_loss_fn = perceptual_loss_fn

        self.ckpt_cb = CheckpointCallback(self.ckpt_path(), self.final_path(),
                                           every_n=cfg.training.checkpoint_every)
        self.sample_cb = SampleGridCallback(os.path.join(cfg.samples_dir, "vae"), "vae")

        self.try_resume(self.device)

    def train_one_epoch(self, epoch):
        self.model.train()
        running = {"total": 0.0, "recon": 0.0, "kld": 0.0}
        w = kl_weight_schedule(epoch, self.cfg.training.kl_warmup_epochs)

        for x in self.loader:
            x = x.to(self.device)
            recon, mu, logvar = self.model(x)
            loss, r, k, _ = vae_loss(
                recon, x, mu, logvar, kld_weight=w,
                perceptual_loss_fn=self.perceptual_loss_fn,
                perceptual_weight=self.cfg.training.perceptual_weight if self.cfg.training.use_perceptual_loss else 0.0,
            )
            self.opt.zero_grad()
            loss.backward()
            self.opt.step()
            running["total"] += loss.item()
            running["recon"] += r.item()
            running["kld"] += k.item()

        n = len(self.loader)
        for key in running:
            running[key] /= n
        running["kl_w"] = w
        return running

    def on_epoch_end(self, epoch, metrics):
        self.logger.info(
            f"epoch {epoch + 1}/{self.total_epochs}  total={metrics['total']:.2f}  "
            f"recon={metrics['recon']:.2f}  kld={metrics['kld']:.2f}  kl_w={metrics['kl_w']:.2f}"
        )
        self.model.eval()
        with torch.no_grad():
            samples = self.model.sample(16, self.device)
        self.sample_cb.maybe_save(epoch, self.total_epochs, samples)
        state = self.state_dict()
        state["history"] = self.history
        self.ckpt_cb.maybe_save(epoch, self.total_epochs, state)

    def state_dict(self):
        return {"model": self.model.state_dict(), "optimizer": self.opt.state_dict()}

    def load_state_dict(self, ckpt):
        self.model.load_state_dict(ckpt["model"])
        self.opt.load_state_dict(ckpt["optimizer"])

    def save_final(self):
        save_checkpoint(self.final_path(), model=self.model.state_dict(), config=str(self.cfg))


def train_vae(cfg, loader=None, perceptual_loss_fn=None):
    trainer = VAETrainer(cfg, loader=loader, perceptual_loss_fn=perceptual_loss_fn)
    history = trainer.run()
    return trainer.model, history

In [ ]:
%%writefile src/training/trainer.py
"""
Generic epoch-loop trainer shared by VAETrainer and DDPMTrainer: handles the
resume-from-checkpoint logic once, so train_vae.py / train_ddpm.py only need
to implement the actual per-batch step.
"""
import os

from src.utils.checkpoint import load_checkpoint
from src.utils.logger import get_logger


class BaseTrainer:
    """Subclasses implement `train_one_epoch(epoch) -> dict_of_metrics` and
    `state_dict()` / `load_state_dict()` for resumability."""

    name = "base"

    def __init__(self, cfg, total_epochs: int):
        self.cfg = cfg
        self.total_epochs = total_epochs
        self.logger = get_logger(self.name, log_dir=cfg.logs_dir)
        self.history = []
        self.start_epoch = 0

    def ckpt_path(self) -> str:
        return os.path.join(self.cfg.checkpoint_dir, f"{self.name}_checkpoint.pt")

    def final_path(self) -> str:
        return os.path.join(self.cfg.checkpoint_dir, f"{self.name}_best.pt")

    def try_resume(self, device):
        path = self.ckpt_path()
        if os.path.exists(path):
            ckpt = load_checkpoint(path, map_location=device)
            self.load_state_dict(ckpt)
            self.history = ckpt["history"]
            self.start_epoch = ckpt["epoch"] + 1
            self.logger.info(f"Resuming {self.name} training from epoch {self.start_epoch}")

    def run(self):
        if self.start_epoch >= self.total_epochs:
            self.logger.info(f"{self.name}: checkpoint already covers requested epochs, skipping.")
            return self.history

        for epoch in range(self.start_epoch, self.total_epochs):
            metrics = self.train_one_epoch(epoch)
            self.history.append(metrics)
            self.on_epoch_end(epoch, metrics)

        self.save_final()
        return self.history

    # --- subclasses must implement these ---
    def train_one_epoch(self, epoch):
        raise NotImplementedError

    def on_epoch_end(self, epoch, metrics):
        raise NotImplementedError

    def state_dict(self):
        raise NotImplementedError

    def load_state_dict(self, ckpt):
        raise NotImplementedError

    def save_final(self):
        raise NotImplementedError

In [ ]:
%%writefile src/utils/__init__.py

In [ ]:
%%writefile src/utils/checkpoint.py
"""Checkpoint save/load helpers."""
import os

import torch


def save_checkpoint(path: str, **kwargs):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(kwargs, path)


def load_checkpoint(path: str, map_location=None) -> dict:
    return torch.load(path, map_location=map_location)

In [ ]:
%%writefile src/utils/device.py
"""Device selection."""
import torch


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
%%writefile src/utils/helpers.py
"""Small stateless helpers used across modules."""
import torch


def kl_weight_schedule(epoch: int, warmup_epochs: int, max_w: float = 1.0) -> float:
    """Linear KL-annealing schedule: 0 -> max_w over `warmup_epochs` epochs."""
    return max_w * min(1.0, epoch / max(1, warmup_epochs))


def count_parameters(model: torch.nn.Module) -> float:
    """Returns parameter count in millions."""
    return sum(p.numel() for p in model.parameters()) / 1e6

In [ ]:
%%writefile src/utils/image_utils.py
"""Image tensor conversion helpers, shared across training/sampling/evaluation."""
import torch
from torchvision.utils import make_grid


def denorm(x: torch.Tensor) -> torch.Tensor:
    """[-1, 1] float tensor -> [0, 1] float tensor, clamped."""
    return (x.clamp(-1, 1) + 1) / 2


def to_uint8(x: torch.Tensor) -> torch.Tensor:
    """[-1, 1] float tensor -> [0, 255] uint8 tensor (for FID / Inception Score)."""
    return (denorm(x) * 255).round().to(torch.uint8)


def to_grid(x: torch.Tensor, nrow: int = 4) -> torch.Tensor:
    """[-1, 1] float batch -> a single [0,1] grid image tensor (C,H,W)."""
    return make_grid(denorm(x), nrow=nrow)

In [ ]:
%%writefile src/utils/logger.py
"""Minimal structured logger (stdlib logging, no external deps)."""
import logging
import os
import sys


def get_logger(name: str = "gen_ai", log_dir: str = None) -> logging.Logger:
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger  # already configured
    logger.setLevel(logging.INFO)

    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(name)s: %(message)s",
                             datefmt="%H:%M:%S")

    stream = logging.StreamHandler(sys.stdout)
    stream.setFormatter(fmt)
    logger.addHandler(stream)

    if log_dir:
        os.makedirs(log_dir, exist_ok=True)
        file_handler = logging.FileHandler(os.path.join(log_dir, f"{name}.log"))
        file_handler.setFormatter(fmt)
        logger.addHandler(file_handler)

    return logger

In [ ]:
%%writefile src/utils/paths.py
"""Filesystem path helpers."""
import os


def ensure_dir(path: str) -> str:
    os.makedirs(path, exist_ok=True)
    return path


def ensure_output_dirs(cfg):
    """Creates the standard outputs/ subfolder layout for a given Config."""
    for sub in ["checkpoints", "logs", "samples", "reconstructions",
                "interpolation", "latent_space", "denoising",
                "training_curves", "gifs"]:
        ensure_dir(os.path.join(cfg.output_dir, sub))

In [ ]:
%%writefile src/utils/seed.py
"""Reproducibility."""
import random

import numpy as np
import torch


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
%%writefile src/visualization/__init__.py

In [ ]:
%%writefile src/visualization/denoising.py
"""Visualizes the DDPM reverse-process trajectory (x_T noise -> x_0 image) as
a strip of intermediate snapshots — makes the "iterative denoising" story
concrete for the report, rather than just showing the final sample."""
import matplotlib.pyplot as plt
import torch

from src.sampling.ddpm_sampler import ddpm_sample
from src.utils.image_utils import denorm


@torch.no_grad()
def save_denoising_strip(unet, diffusion, img_size, device, save_path):
    _, trajectory = ddpm_sample(unet, diffusion, img_size, batch_size=1, device=device,
                                 return_trajectory=True)
    frames = [denorm(t[0]).permute(1, 2, 0).numpy() for t in trajectory]

    fig, axes = plt.subplots(1, len(frames), figsize=(2 * len(frames), 2))
    for i, frame in enumerate(frames):
        axes[i].imshow(frame)
        axes[i].axis("off")
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close()
    return frames

In [ ]:
%%writefile src/visualization/gif.py
"""Turns a list of image frames (e.g. from denoising.py) into an animated GIF."""
import numpy as np


def save_gif(frames, save_path, duration_ms=80):
    import imageio

    uint8_frames = [(np.clip(f, 0, 1) * 255).astype(np.uint8) for f in frames]
    imageio.mimsave(save_path, uint8_frames, duration=duration_ms / 1000.0, loop=0)

In [ ]:
%%writefile src/visualization/interpolation.py
"""Latent-space interpolation grid: decode a linear path between two random
latent vectors — a classic qualitative check that the VAE latent space is
smooth/meaningful rather than degenerate."""
import matplotlib.pyplot as plt
import torch

from src.sampling.latent_sampler import interpolate_latents
from src.utils.image_utils import denorm


@torch.no_grad()
def save_interpolation_grid(vae, device, save_path, steps=10, seed=0):
    g = torch.Generator(device=device).manual_seed(seed)
    z_start = torch.randn(1, vae.latent_dim, device=device, generator=g)[0]
    z_end = torch.randn(1, vae.latent_dim, device=device, generator=g)[0]

    imgs = interpolate_latents(vae, z_start, z_end, steps=steps)

    fig, axes = plt.subplots(1, steps, figsize=(2 * steps, 2))
    for i in range(steps):
        axes[i].imshow(denorm(imgs[i]).cpu().permute(1, 2, 0).numpy())
        axes[i].axis("off")
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close()

In [ ]:
%%writefile src/visualization/latent_space.py
"""2D projection (PCA) of VAE latent codes for a batch of real images — a
sanity check that the learned latent space has meaningful structure rather
than collapsing to a single point (posterior collapse)."""
import matplotlib.pyplot as plt
import torch


@torch.no_grad()
def save_latent_space_plot(vae, real_batch, device, save_path):
    from sklearn.decomposition import PCA

    vae.eval()
    x = real_batch.to(device)
    mu, _ = vae.encode(x)
    mu = mu.cpu().numpy()

    if mu.shape[1] > 2:
        mu_2d = PCA(n_components=2).fit_transform(mu)
    else:
        mu_2d = mu

    plt.figure(figsize=(6, 6))
    plt.scatter(mu_2d[:, 0], mu_2d[:, 1], alpha=0.6, s=15)
    plt.title("VAE latent space (PCA projection)")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close()

In [ ]:
%%writefile src/visualization/reconstruction.py
"""Side-by-side real vs. VAE-reconstructed image grid — visual complement to
src/evaluation/reconstruction.py's numeric MSE/PSNR."""
import matplotlib.pyplot as plt
import torch

from src.utils.image_utils import denorm


@torch.no_grad()
def save_reconstruction_grid(vae, real_batch, device, save_path, n=8):
    vae.eval()
    x = real_batch[:n].to(device)
    recon, _, _ = vae(x)

    fig, axes = plt.subplots(2, n, figsize=(2 * n, 4))
    for i in range(n):
        axes[0, i].imshow(denorm(x[i]).cpu().permute(1, 2, 0).numpy())
        axes[0, i].axis("off")
        axes[1, i].imshow(denorm(recon[i]).cpu().permute(1, 2, 0).numpy())
        axes[1, i].axis("off")
    axes[0, 0].set_ylabel("Real", fontsize=12)
    axes[1, 0].set_ylabel("Reconstructed", fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close()

In [ ]:
%%writefile src/visualization/sampling_grid.py
"""Real / VAE / DDPM sample-grid visualizations. Uses whichever matplotlib
backend is already active (inline in a notebook, Agg/etc. in a script) —
saving via plt.savefig() works regardless, so we don't force a backend here."""
import matplotlib.pyplot as plt

from src.utils.image_utils import to_grid


def save_grid(imgs, title, save_path, n=16, nrow=4):
    grid = to_grid(imgs[:n], nrow=nrow)
    plt.figure(figsize=(6, 6))
    plt.axis("off")
    plt.title(title)
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close()


def save_comparison_grid(real, vae_imgs, ddpm_imgs, save_path, n=9):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, imgs, title in zip(axes, [real, vae_imgs, ddpm_imgs], ["Real", "VAE", "DDPM"]):
        grid = to_grid(imgs[:n], nrow=3)
        ax.imshow(grid.permute(1, 2, 0).numpy())
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close()

In [ ]:
%%writefile src/visualization/training_curves.py
"""Plots and saves loss curves for VAE / DDPM training history."""
import matplotlib.pyplot as plt


def save_curve_plot(history_keys_to_values: dict, title: str, save_path: str, xlabel="epoch"):
    plt.figure(figsize=(7, 5))
    for label, values in history_keys_to_values.items():
        plt.plot(values, label=label)
    plt.legend()
    plt.title(title)
    plt.xlabel(xlabel)
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close()


def save_vae_curves(history: list, save_path: str):
    save_curve_plot(
        {"recon": [h["recon"] for h in history], "kld": [h["kld"] for h in history]},
        "VAE training curves", save_path,
    )


def save_ddpm_curves(history: list, save_path: str):
    save_curve_plot({"loss": [h["loss"] for h in history]}, "DDPM training loss", save_path)

In [ ]:
%%writefile tests/__init__.py

In [ ]:
%%writefile tests/test_dataset.py
"""Sanity tests for the CelebA dataset pipeline using a synthetic temp dataset
(does not require the real CelebA download)."""
import os
import tempfile

import torch
from PIL import Image

from src.datasets.dataset import CelebADataset
from src.datasets.transforms import build_transform


def _make_fake_images(n=5, size=200):
    tmp_dir = tempfile.mkdtemp()
    for i in range(n):
        img = Image.fromarray((torch.rand(size, size, 3) * 255).byte().numpy())
        img.save(os.path.join(tmp_dir, f"{i:06d}.jpg"))
    return tmp_dir


def test_dataset_returns_correct_tensor_shape():
    tmp_dir = _make_fake_images(n=3)
    files = [os.path.join(tmp_dir, f) for f in sorted(os.listdir(tmp_dir))]
    transform = build_transform(img_size=64, augment=False)
    dataset = CelebADataset(files, transform)

    assert len(dataset) == 3
    x = dataset[0]
    assert x.shape == (3, 64, 64)
    assert x.min() >= -1.0 and x.max() <= 1.0

In [ ]:
%%writefile tests/test_losses.py
"""Sanity tests for loss functions."""
import torch

from src.losses.diffusion_loss import simple_diffusion_loss
from src.losses.vae_loss import vae_loss


def test_simple_diffusion_loss():
    pred = torch.randn(4, 3, 16, 16)
    true = torch.randn(4, 3, 16, 16)
    loss = simple_diffusion_loss(pred, true)
    assert torch.isfinite(loss)


def test_vae_loss_components():
    recon = torch.randn(4, 3, 16, 16)
    x = torch.randn(4, 3, 16, 16)
    mu = torch.randn(4, 8)
    logvar = torch.randn(4, 8)
    total, recon_l, kld, perceptual = vae_loss(recon, x, mu, logvar, kld_weight=0.5)
    assert torch.isfinite(total)
    assert torch.isfinite(recon_l)
    assert torch.isfinite(kld)

In [ ]:
%%writefile tests/test_sampler.py
"""Sanity tests for DDPM and DDIM samplers."""
import torch

from src.models.diffusion import GaussianDiffusion
from src.models.unet import UNet
from src.sampling.ddim_sampler import ddim_sample
from src.sampling.ddpm_sampler import ddpm_sample


def _tiny_setup():
    model = UNet(img_channels=3, base_ch=8, ch_mults=(1, 2), time_dim=32)
    diffusion = GaussianDiffusion(timesteps=10, device="cpu")
    return model, diffusion


def test_ddpm_sample_shape():
    model, diffusion = _tiny_setup()
    out = ddpm_sample(model, diffusion, img_size=32, batch_size=2, device="cpu")
    assert out.shape == (2, 3, 32, 32)


def test_ddim_sample_shape_and_fewer_steps():
    model, diffusion = _tiny_setup()
    out = ddim_sample(model, diffusion, img_size=32, batch_size=2, device="cpu", ddim_steps=5)
    assert out.shape == (2, 3, 32, 32)

In [ ]:
%%writefile tests/test_unet.py
"""Sanity tests for the U-Net + diffusion process: shapes, finite loss,
correct sampling output shape."""
import torch

from src.models.diffusion import GaussianDiffusion
from src.models.unet import UNet


def test_unet_forward_shape():
    model = UNet(img_channels=3, base_ch=8, ch_mults=(1, 2), time_dim=32)
    x = torch.randn(2, 3, 32, 32)
    t = torch.randint(0, 10, (2,))
    out = model(x, t)
    assert out.shape == x.shape


def test_diffusion_training_loss_is_finite():
    model = UNet(img_channels=3, base_ch=8, ch_mults=(1, 2), time_dim=32)
    diffusion = GaussianDiffusion(timesteps=10, device="cpu")
    x0 = torch.randn(2, 3, 32, 32)
    loss = diffusion.training_loss(model, x0)
    assert torch.isfinite(loss)
    assert loss.dim() == 0

In [ ]:
%%writefile tests/test_vae.py
"""Sanity tests for the VAE: correct output shapes, valid loss, working sample()."""
import torch

from src.losses.vae_loss import vae_loss
from src.models.vae import VAE


def test_vae_forward_shapes():
    x = torch.randn(4, 3, 64, 64)
    model = VAE(img_channels=3, base_ch=16, latent_dim=32, img_size=64)
    recon, mu, logvar = model(x)
    assert recon.shape == x.shape
    assert mu.shape == (4, 32)
    assert logvar.shape == (4, 32)


def test_vae_loss_is_finite_scalar():
    x = torch.randn(4, 3, 64, 64)
    model = VAE(img_channels=3, base_ch=16, latent_dim=32, img_size=64)
    recon, mu, logvar = model(x)
    total, recon_l, kld, perceptual = vae_loss(recon, x, mu, logvar)
    assert torch.isfinite(total)
    assert total.dim() == 0


def test_vae_sample_shape():
    model = VAE(img_channels=3, base_ch=16, latent_dim=32, img_size=64)
    samples = model.sample(5, "cpu")
    assert samples.shape == (5, 3, 64, 64)

In [ ]:
%%writefile train.py
#!/usr/bin/env python3
"""
Train the VAE then the DDPM on CelebA.

    python train.py

Adjust hyperparameters by editing configs/*.py, or by constructing a
Config(...) with overrides in your own script / notebook.
"""
from configs.config import Config
from src.engine.trainer import GenAITrainer
from src.visualization.training_curves import save_ddpm_curves, save_vae_curves


def main():
    cfg = Config()
    trainer = GenAITrainer(cfg)
    result = trainer.run()

    save_vae_curves(result["vae_history"], f"{cfg.output_dir}/training_curves/vae_curves.png")
    save_ddpm_curves(result["ddpm_history"], f"{cfg.output_dir}/training_curves/ddpm_curves.png")

    print("Done. Checkpoints saved to", cfg.checkpoint_dir)


if __name__ == "__main__":
    main()

## 2. Verify the file tree, then run the pipeline

In [ ]:
!find /kaggle/working/Gen_AI -name "*.py" | sort

In [ ]:
# Quick sanity check with pytest before spending GPU time on real training
!python -m pytest tests/ -q

## 3. Point the dataset path if needed

If `train.py` raises `FileNotFoundError` for CelebA, run the diagnostic below
and add the printed path to `CELEBA_DIR_CANDIDATES` in
`src/datasets/preprocessing.py` (re-run that `%%writefile` cell after
editing), or just set `cfg.dataset.data_dir` directly in the next cell.

In [ ]:
import os
for root_name in os.listdir("/kaggle/input"):
    root = os.path.join("/kaggle/input", root_name)
    if "celeba" in root_name.lower():
        for dirpath, dirnames, filenames in os.walk(root):
            depth = dirpath.replace(root, "").count(os.sep)
            if depth <= 2:
                print(dirpath, "->", dirnames[:5], filenames[:3])

## 4. Smoke test (small config) before the real run

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/Gen_AI")
from configs.config import Config
from src.engine.trainer import GenAITrainer
from src.engine.evaluator import GenAIEvaluator

cfg = Config()
cfg.output_dir = "/kaggle/working/Gen_AI/outputs"
# cfg.dataset.data_dir = "/kaggle/input/<paste-exact-path-here>"  # uncomment if needed
cfg.dataset.subset_size = 2000
cfg.training.vae_epochs = 2
cfg.training.ddpm_epochs = 2
cfg.n_eval_samples = 100

trainer = GenAITrainer(cfg)
result = trainer.run()

evaluator = GenAIEvaluator(cfg)
results, real_imgs, vae_imgs, ddpm_imgs = evaluator.run(
    vae=result["vae"], unet=result["unet"], diffusion=result["diffusion"], ema=result["ema"],
    files=result["files"], transform=result["eval_transform"],
)
print(results)

## 5. Full run (once the smoke test passes)

Either edit `configs/training.py` / `configs/dataset.py` directly (Kaggle's
file editor pane) and run `!python train.py` + `!python evaluate.py`, or just
override the Config in Python as below.

In [ ]:
cfg = Config()
cfg.output_dir = "/kaggle/working/Gen_AI/outputs"
# cfg.dataset.data_dir = "/kaggle/input/<paste-exact-path-here>"  # uncomment if needed
cfg.dataset.subset_size = 30000
cfg.training.vae_epochs = 30
cfg.training.ddpm_epochs = 40
cfg.n_eval_samples = 2000

trainer = GenAITrainer(cfg)
result = trainer.run()

In [ ]:
evaluator = GenAIEvaluator(cfg)
results, real_imgs, vae_imgs, ddpm_imgs = evaluator.run(
    vae=result["vae"], unet=result["unet"], diffusion=result["diffusion"], ema=result["ema"],
    files=result["files"], transform=result["eval_transform"],
)
print(results)

## 6. Extra visualizations + download

Reconstruction grid, latent-space PCA, latent interpolation, denoising-process
GIF — all saved under `outputs/`.

In [ ]:
from src.visualization.reconstruction import save_reconstruction_grid
from src.visualization.latent_space import save_latent_space_plot
from src.visualization.interpolation import save_interpolation_grid
from src.visualization.denoising import save_denoising_strip
from src.visualization.gif import save_gif

vae, device = result["vae"], evaluator.device
from src.models.unet import UNet
eval_unet = UNet.from_config(cfg).to(device)
result["ema"].copy_to(eval_unet)

save_reconstruction_grid(vae, real_imgs, device, f"{cfg.output_dir}/reconstructions/vae_reconstructions.png")
save_latent_space_plot(vae, real_imgs, device, f"{cfg.output_dir}/latent_space/vae_latent_pca.png")
save_interpolation_grid(vae, device, f"{cfg.output_dir}/interpolation/vae_interpolation.png")
frames = save_denoising_strip(eval_unet, result["diffusion"], cfg.dataset.img_size, device,
                               f"{cfg.output_dir}/denoising/ddpm_denoising_strip.png")
save_gif(frames, f"{cfg.output_dir}/gifs/ddpm_denoising.gif")
print("All visualizations saved.")

In [ ]:
import shutil, os
from IPython.display import FileLink, display

shutil.make_archive("/kaggle/working/Gen_AI_results", "zip", cfg.output_dir)
display(FileLink("/kaggle/working/Gen_AI_results.zip"))

## 7. Push to GitHub

The `configs/`, `src/`, `tests/`, and top-level `.py` files written above are
byte-for-byte the repo files. From your own machine (or a Kaggle terminal
with internet + git configured):

```bash
cd Gen_AI
git init
git add .
git commit -m "VAE vs DDPM from scratch"
git remote add origin <your-repo-url>
git push -u origin main
```